# Stage 3C: Binary Tumor Classification (Extended CV Pipeline)

**Fixed version** — all 6 issues from code review resolved + 5 additional fixes (temperature collapse, threshold inf, collapsed-fold exclusion, post-calibration guard, dead-code removal).

| # | Severity | Issue | Fix |
|---|----------|-------|-----|
| 1 | CRITICAL | `final_retrain_and_test` ROI-row val split leaks patients | Patient-safe split via `_patient_safe_val_split()` |
| 2 | CRITICAL | Header claimed focal loss / MixUp / label smoothing not implemented | Docstring rewritten to match actual code |
| 3 | IMPORTANT | Docs/logs said 'patient-level evaluation'; actually image-level | All comments, titles, logs updated to 'image-level' |
| 4 | MINOR | ECE last bin excluded prob==1.0 | Last bin now closed `[lo, 1.0]` |
| 5 | MINOR | Image label via majority-vote not documented as design choice | Explicit comment added |
| 6 | MINOR | Calibration plot used ROI-level probs but labelled as image-level | Now plots both ROI-level and image-level panels |

**Data paths:**
- ROIs: `/kaggle/input/datasets/sadibhasan/roi-dataset/{split}/{class}/`
- Metadata: `/kaggle/input/datasets/blaster21/5foldmetadata/roi_metadata_with_patient.csv`


## Module Docstring & Imports


In [1]:
"""
STAGE 3C: BINARY TUMOR CLASSIFICATION — EXTENDED CV PIPELINE
==============================================================================
Dataset  : BTXRD ROI Dataset (sadibhasan / blaster21, Kaggle)

Imbalance handling strategy:
  1. WeightedRandomSampler  (pure 1/frequency → ~50/50 batches)
  2. CrossEntropyLoss with label_smoothing=0.1  (no class weights)
  3. Class-balanced early stopping  (monitors AUC-PR, not accuracy)
  4. Full per-class diagnostics printed at startup

Regularisation strategy (anti-overfitting):
  • Label smoothing 0.1  — prevents logit saturation / memorisation
  • Weight decay 5e-3    — L2 penalty on full backbone weights
  • Finetune LR 1e-5     — 50× lower than head LR; backbone moves slowly
  • Unfreeze at epoch 15 — head converges on frozen features first
  • BatchNorm frozen after unfreeze — preserves ImageNet BN statistics
  • Head dropout 0.6/0.5 — increased from 0.5/0.4
  • MixUp alpha=0.4      — active for with_aug condition only

Backbones:
  DenseNet121-SE | ResNet18-SE | EfficientNet-B0-SE | MobileNetV2-SE
  ConvNeXt-Tiny-SE | EfficientNet-B2-SE

CV:  5-Fold Stratified Group KFold (group=patient_id, zero leakage)
     × 2 aug conditions (with_aug / no_aug)

Aggregation note:
  source_image is set directly from roi_filename in the metadata CSV —
  no regex stripping of box/iou/conf suffixes is applied.
  This means each ROI file maps to a unique source_image key (1:1).
  aggregate_rois() still pools by source_image key before computing metrics;
  with 1 ROI per image the pooling is a no-op (max of a single value).
  If multiple ROIs per X-ray share the same filename prefix and you want
  them pooled, add a dedicated source_image column to the metadata CSV and
  update load_manifest_from_metadata() to read it.
  Evaluation granularity is IMAGE-level, not patient-level.
==============================================================================
"""


'\nSTAGE 3C: BINARY TUMOR CLASSIFICATION — EXTENDED CV PIPELINE\n==============================================================================\nDataset  : BTXRD ROI Dataset (sadibhasan / blaster21, Kaggle)\n\nImbalance handling strategy:\n  1. WeightedRandomSampler  (pure 1/frequency → ~50/50 batches)\n  2. CrossEntropyLoss with label_smoothing=0.1  (no class weights)\n  3. Class-balanced early stopping  (monitors AUC-PR, not accuracy)\n  4. Full per-class diagnostics printed at startup\n\nRegularisation strategy (anti-overfitting):\n  • Label smoothing 0.1  — prevents logit saturation / memorisation\n  • Weight decay 5e-3    — L2 penalty on full backbone weights\n  • Finetune LR 1e-5     — 50× lower than head LR; backbone moves slowly\n  • Unfreeze at epoch 15 — head converges on frozen features first\n  • BatchNorm frozen after unfreeze — preserves ImageNet BN statistics\n  • Head dropout 0.6/0.5 — increased from 0.5/0.4\n  • MixUp alpha=0.4      — active for with_aug condition only

## IMPORTS


In [2]:

import os
import json
import random
import warnings
from copy import deepcopy
from pathlib import Path
from collections import Counter, defaultdict
from itertools import product as it_product

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    confusion_matrix, precision_recall_fscore_support,
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    matthews_corrcoef, brier_score_loss,
    classification_report,
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import StratifiedGroupKFold

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from PIL import Image, ImageOps, UnidentifiedImageError
from tqdm import tqdm

warnings.filterwarnings('ignore')


## CONFIGURATION


In [3]:

class Config:

    # ── Dataset root ──────────────────────────────────────────────────────
    # ROI images are stored as:
    #   {ROI_DATASET_DIR}/{split}/{class}/{filename}
    ROI_DATASET_DIR = "/kaggle/input/datasets/sadibhasan/roi-dataset"

    # Metadata CSV — authoritative source for patient_id, split, class, etc.
    # Columns used: roi_filename, split, class, label, patient_id
    METADATA_CSV = (
        "/kaggle/input/datasets/blaster21/5foldmetadata/"
        "roi_metadata_with_patient.csv"
    )

    RESULTS_DIR  = "/kaggle/working/results_stage3c_cv"
    RANDOM_SEED  = 42

    # ── Image ─────────────────────────────────────────────────────────────
    IMAGE_SIZE = 256

    # ── Training ──────────────────────────────────────────────────────────
    BATCH_SIZE                  = 32
    CV_EPOCHS                   = 40
    FINAL_EPOCHS                = 50
    LEARNING_RATE               = 5e-4
    # Finetune LR is deliberately 50× lower than head LR so the pretrained
    # backbone moves very slowly after unfreezing — the main lever against
    # post-unfreeze memorisation.
    FINETUNE_LR                 = 1e-5
    # Higher weight decay provides L2 penalty across the full backbone;
    # 5e-3 is standard for fine-tuning on small medical datasets.
    WEIGHT_DECAY                = 5e-3
    GRAD_CLIP                   = 1.0
    GRADIENT_ACCUMULATION_STEPS = 2
    # Delay unfreeze so the classification head fully converges on frozen
    # features before the backbone starts moving.
    UNFREEZE_EPOCH              = 15
    NUM_WORKERS                 = 4

    # ── Regularisation ────────────────────────────────────────────────────
    # Label smoothing: prevents the model from pushing logits to ±∞ and
    # memorising training labels.  0.1 is the standard value.
    LABEL_SMOOTHING             = 0.1
    # Dropout rates for all classifier heads (applied consistently across
    # all backbone variants so comparisons are fair).
    HEAD_DROPOUT_1              = 0.6    # first dropout (was 0.5)
    HEAD_DROPOUT_2              = 0.5    # second dropout (was 0.4)
    # MixUp alpha: only active for with_aug condition.
    # Interpolates pairs of training images + labels, preventing memorisation.
    MIXUP_ALPHA                 = 0.4

    # ── CV ────────────────────────────────────────────────────────────────
    N_FOLDS = 5

    # ── Class imbalance ──────────────────────────────────────────────────
    USE_WEIGHTED_SAMPLER = True

    # ── SE block ──────────────────────────────────────────────────────────
    SE_REDUCTION = 16

    # ── Warmup ────────────────────────────────────────────────────────────
    USE_WARMUP    = True
    WARMUP_EPOCHS = 5

    # ── Early stopping ────────────────────────────────────────────────────
    MONITOR_METRIC      = 'auc_pr'
    EARLY_STOP_PATIENCE = 10
    MIN_DELTA           = 1e-4
    MIN_EPOCHS          = 20

    # ── EMA ───────────────────────────────────────────────────────────────
    USE_EMA   = True
    EMA_ALPHA = 0.3

    # ── Class mapping ─────────────────────────────────────────────────────
    CLASS_NAMES     = ['Benign', 'Malignant']
    NUM_CLASSES     = 2
    LABEL_TO_CLASS  = {0: 'benign', 1: 'malignant'}
    CLASS_TO_LABEL  = {'benign': 0, 'malignant': 1}

    CALIB_N_BINS = 10

    MODEL_NAMES = [
        'densenet121_se',
        'resnet18_se',
        'efficientnet_b0_se',
        'mobilenet_v2_se',
        'convnext_tiny_se',
        'efficientnet_b2_se',
    ]

    # ── Final retrain internal-val fraction ───────────────────────────────
    # Fraction of PATIENTS (not ROIs) held out as internal val for early
    # stopping / threshold calibration.  Patient-safe split.
    FINAL_VAL_PATIENT_FRACTION = 0.10


cfg = Config()


## UTILS — seed, encoder, path normalisation


In [4]:

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.ndarray):  return obj.tolist()
        if isinstance(obj, np.integer):  return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.bool_):    return bool(obj)
        return super().default(obj)


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED']       = str(seed)


set_seed(cfg.RANDOM_SEED)


def normalize_img_key(img_id: str) -> str:
    return Path(img_id).name


## SOURCE IMAGE EXTRACTION


In [5]:

import re as _re

_FRCNN_PATTERN = _re.compile(
    r'^(.+?)_box\d+_iou\d+_conf\d+',
    _re.IGNORECASE
)
_ROI_PATTERN = _re.compile(r'^(.+?)_roi\d*$', _re.IGNORECASE)


def extract_source_image(roi_filename: str) -> str:
    """
    Extract the original image ID from a ROI filename.

    Supported patterns (in priority order):
      1. IMG000222_box000_iou65_conf84.jpg  →  IMG000222.jpg
      2. IMG000222_roi0.png                 →  IMG000222.png
      3. Anything else                      →  full filename unchanged
    """
    fname = Path(roi_filename).name
    stem  = Path(roi_filename).stem
    ext   = Path(roi_filename).suffix or '.jpg'

    m = _FRCNN_PATTERN.match(stem)
    if m:
        return m.group(1) + ext

    m = _ROI_PATTERN.match(stem)
    if m:
        return m.group(1) + ext

    return fname


# ============================================================================
# MANIFEST LOADER
# Primary source: METADATA_CSV (contains roi_filename, split, class, patient_id)
# Constructs full roi_filepath from ROI_DATASET_DIR/{split}/{class}/{roi_filename}
# ============================================================================

def _normalise_class(val):
    v = str(val).strip().lower()
    if v in ('malignant', '1', 'mal'):
        return 'malignant'
    if v in ('benign', '0', 'ben'):
        return 'benign'
    return v   # returned as-is so audit_dataframe can reject it


def load_manifest_from_metadata(metadata_csv: str) -> pd.DataFrame:
    """
    Load the authoritative metadata CSV and build a DataFrame with columns:
      roi_filepath, class, split, source_image, patient_id

    The metadata CSV must have at minimum:
      roi_filename  — bare filename (e.g. IMG000346_box000_iou72_conf89.jpg)
      split         — train / val / test
      class         — benign / malignant  (or label column as int)
      patient_id    — patient identifier
    """
    df = pd.read_csv(metadata_csv)
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]

    # ── Resolve 'class' column ────────────────────────────────────────────
    if 'class' not in df.columns:
        if 'label' in df.columns:
            df['class'] = df['label'].apply(_normalise_class)
        else:
            raise ValueError(
                f"Metadata CSV has no 'class' or 'label' column.\n"
                f"Found: {list(df.columns)}")
    else:
        df['class'] = df['class'].apply(_normalise_class)

    # ── Resolve filename column ───────────────────────────────────────────
    fname_col = next(
        (c for c in ['roi_filename', 'filename', 'file', 'roi_path',
                      'filepath', 'roi_filepath', 'image_name']
         if c in df.columns), None)
    if fname_col is None:
        raise ValueError(
            f"Cannot find ROI filename column in metadata CSV.\n"
            f"Found: {list(df.columns)}")
    df.rename(columns={fname_col: '_roi_filename'}, inplace=True)

    # ── Build full roi_filepath ───────────────────────────────────────────
    def _build_path(row):
        fname = Path(str(row['_roi_filename'])).name
        split = str(row.get('split', 'train')).strip().lower()
        cls   = str(row.get('class',  'benign')).strip().lower()
        # Primary candidate
        p = os.path.join(cfg.ROI_DATASET_DIR, split, cls, fname)
        if os.path.exists(p):
            return p
        # Search all combos (handles mislabelled split/class in metadata)
        for sp in ('train', 'val', 'test'):
            for cl in ('benign', 'malignant'):
                alt = os.path.join(cfg.ROI_DATASET_DIR, sp, cl, fname)
                if os.path.exists(alt):
                    return alt
        return p   # return primary so audit catches it as missing

    df['roi_filepath'] = df.apply(_build_path, axis=1)

    # ── source_image ──────────────────────────────────────────────────────
    # When loading from the authoritative metadata CSV, roi_filename IS the
    # exact filename that exists on disk in {ROI_DATASET_DIR}/{split}/{class}/.
    # We use it directly as the grouping key — no regex stripping of box/iou/conf
    # suffixes.  extract_source_image() is only used in the folder-scan fallback
    # where the metadata is absent.
    df['source_image'] = df['_roi_filename'].apply(lambda x: Path(str(x)).name)

    # ── patient_id ────────────────────────────────────────────────────────
    pid_col = next(
        (c for c in ['patient_id', 'patientid', 'patient', 'pid',
                      'subject_id', 'case_id']
         if c in df.columns), None)
    if pid_col and pid_col != 'patient_id':
        df.rename(columns={pid_col: 'patient_id'}, inplace=True)
    if 'patient_id' not in df.columns:
        print("  ⚠️  No patient_id column — using source_image stem as fallback.")
        df['patient_id'] = df['source_image'].apply(lambda x: Path(x).stem)

    keep = ['roi_filepath', 'class', 'split', 'source_image', 'patient_id']
    return df[keep].copy()


def load_all_splits():
    """
    Load train/val/test DataFrames from METADATA_CSV.
    Falls back to folder scan if metadata is absent.
    """
    if os.path.exists(cfg.METADATA_CSV):
        print(f"  📂 Loading metadata: {cfg.METADATA_CSV}")
        df_all = load_manifest_from_metadata(cfg.METADATA_CSV)
    else:
        print("  ⚠️  Metadata CSV not found — scanning folder structure...")
        df_all = _folder_scan_fallback()

    # Normalise split values
    df_all['split'] = df_all['split'].str.strip().str.lower()

    train_df = df_all[df_all['split'] == 'train'].reset_index(drop=True)
    val_df   = df_all[df_all['split'] == 'val'].reset_index(drop=True)
    test_df  = df_all[df_all['split'] == 'test'].reset_index(drop=True)

    _verify_source_extraction_on_sample(train_df)
    return train_df, val_df, test_df


def _folder_scan_fallback() -> pd.DataFrame:
    rows = []
    for split in ('train', 'val', 'test'):
        for cls in ('benign', 'malignant'):
            folder = os.path.join(cfg.ROI_DATASET_DIR, split, cls)
            if not os.path.isdir(folder):
                continue
            for fname in os.listdir(folder):
                if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                    rows.append({
                        'roi_filepath': os.path.join(folder, fname),
                        'class':        cls,
                        'split':        split,
                        'source_image': extract_source_image(fname),
                        'patient_id':   Path(extract_source_image(fname)).stem,
                    })
    return pd.DataFrame(rows)


def _verify_source_extraction_on_sample(df: pd.DataFrame, n_samples: int = 5):
    """
    When loaded from metadata, source_image == roi_filename (no stripping).
    Prints a mismatch warning if transformation was accidentally applied.
    """
    print("\n  source_image check (should match roi_filepath basename exactly):")
    print(f"  {'roi_filepath basename':<45}  -> source_image")
    print(f"  {'--'*22}      {'--'*17}")
    shown = set()
    for _, row in df.iterrows():
        roi = Path(row['roi_filepath']).name
        src = row['source_image']
        icon = "OK" if roi == src else "!! MISMATCH"
        if roi not in shown:
            print(f"  {roi:<45}  -> {src}  {icon}")
            shown.add(roi)
        if len(shown) >= n_samples:
            break
    print()

def _print_source_image_stats(df: pd.DataFrame, split: str):
    """
    Report ROI / source_image counts.
    When source_image == roi_filename (metadata-loaded data), ratio will be
    1.0 — that is correct and expected, not an error.
    Aggregation for evaluation is driven by patient_id (CV groups) and
    source_image (aggregate_rois pooling); both are printed here.
    """
    n_rois = len(df)
    n_src  = df['source_image'].nunique()
    ratio  = n_rois / max(n_src, 1)
    # ratio==1 is valid when each file has a unique source_image key
    status = "OK (1 ROI per image)" if ratio == 1.0 else f"{ratio:.1f} ROIs/image"
    print(f"  {split:5s}: {n_rois} ROIs / {n_src} source images — {status}")
    if 'patient_id' in df.columns:
        n_pat = df['patient_id'].nunique()
        print(f"  {split:5s}: {n_pat} unique patients  "
              f"({n_src / max(n_pat, 1):.1f} images/patient)")


## DATA AUDIT


In [6]:

def audit_dataframe(df: pd.DataFrame, split: str,
                    check_files: bool = True) -> pd.DataFrame:
    sep = "=" * 70
    print(f"\n  {sep}")
    print(f"  DATA AUDIT [{split.upper()}]  —  {len(df)} rows")
    print(f"  {sep}")

    original_n = len(df)

    # ── Label audit ───────────────────────────────────────────────────────
    valid_labels = {'benign', 'malignant'}
    bad_label_mask = ~df['class'].isin(valid_labels)
    if bad_label_mask.any():
        bad = df.loc[bad_label_mask, 'class'].value_counts().to_dict()
        print(f"  ❌ LABEL AUDIT: {bad_label_mask.sum()} rows with unexpected labels: {bad}")
        print(f"     These rows will be DROPPED.")
        df = df[~bad_label_mask].reset_index(drop=True)
    else:
        print(f"  ✅ Label audit  : all labels are benign/malignant")

    label_counts = df['class'].value_counts().to_dict()
    n_ben = label_counts.get('benign', 0)
    n_mal = label_counts.get('malignant', 0)
    print(f"  ✅ Class counts : benign={n_ben}  malignant={n_mal}  "
          f"ratio={n_ben/max(n_mal,1):.2f}:1")

    # ── File existence + readability check ────────────────────────────────
    if check_files:
        missing, corrupt = [], []
        for idx, row in df.iterrows():
            p = str(row['roi_filepath'])
            if not os.path.exists(p):
                missing.append(idx)
            else:
                try:
                    with Image.open(p) as im:
                        im.verify()
                except Exception:
                    corrupt.append(idx)

        bad_file_idx = missing + corrupt
        if missing:
            print(f"  ❌ FILE AUDIT   : {len(missing)} missing paths — dropped")
        if corrupt:
            print(f"  ❌ FILE AUDIT   : {len(corrupt)} corrupt/unreadable files — dropped")
        if not bad_file_idx:
            print(f"  ✅ File audit   : all {len(df)} files exist and are readable")
        else:
            df = df.drop(index=bad_file_idx).reset_index(drop=True)

    # ── Source-image grouping sanity check ────────────────────────────────
    src_counts = df.groupby('source_image').size()
    max_rois   = src_counts.max()
    mean_rois  = src_counts.mean()
    n_src      = src_counts.shape[0]
    print(f"  ✅ Source images: {n_src} unique  |  "
          f"ROIs/image: mean={mean_rois:.1f}  max={max_rois}")
    if max_rois > 50:
        print(f"  ⚠️  source_image collision? max={max_rois} ROIs — "
              f"check extract_source_image()")

    # ── Sampler balance preview ────────────────────────────────────────────
    labels_arr = [cfg.CLASS_TO_LABEL.get(c, -1) for c in df['class']]
    cnts = Counter(labels_arr)
    if cnts[0] > 0 and cnts[1] > 0:
        sw0 = 1.0 / cnts[0]; sw1 = 1.0 / cnts[1]
        total_w = sw0 * cnts[0] + sw1 * cnts[1]
        pct_ben = 100 * sw0 * cnts[0] / total_w
        pct_mal = 100 * sw1 * cnts[1] / total_w
        print(f"  ✅ Sampler balance: benign≈{pct_ben:.1f}%  malignant≈{pct_mal:.1f}%  "
              f"(target ~50/50)")

    dropped = original_n - len(df)
    if dropped:
        print(f"  ⚠️  Total dropped: {dropped} rows  ({dropped/original_n*100:.1f}%)")
    print(f"  {sep}")
    return df


## CLASS IMBALANCE ANALYSIS


In [7]:

def compute_class_statistics(df: pd.DataFrame,
                              split_name: str = 'train') -> dict:
    counts  = df['class'].value_counts().to_dict()
    n_ben   = counts.get('benign',    0)
    n_mal   = counts.get('malignant', 0)
    n_tot   = n_ben + n_mal

    if n_tot == 0:
        print(f"  ⚠️  [{split_name}] No samples found!")
        return {}

    ratio    = n_ben / max(n_mal, 1)
    mal_freq = n_mal / n_tot
    ben_freq = n_ben / n_tot

    sep = "-" * 60
    print(f"\n  {sep}")
    print(f"  CLASS STATISTICS [{split_name.upper()}]")
    print(f"  {sep}")
    print(f"  Benign    : {n_ben:6d}  ({ben_freq*100:.1f}%)")
    print(f"  Malignant : {n_mal:6d}  ({mal_freq*100:.1f}%)")
    print(f"  Total     : {n_tot:6d}  (B:M ratio {ratio:.2f}:1)")
    print(f"  Loss    : plain CrossEntropyLoss  (no class weights)")
    print(f"  Sampler : WeightedRandomSampler  1/n per class")
    print(f"  Effect  : ~50/50 batches, equal gradient per class")
    print(f"  {sep}")

    return {'n_benign': n_ben, 'n_malignant': n_mal, 'ratio': ratio}


## SE BLOCK


In [8]:

class SEBlock(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Sequential(
            nn.Linear(channels, mid, bias=False), nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False), nn.Sigmoid())

    def forward(self, x):
        b, c = x.shape[:2]
        y = self.pool(x).view(b, c)
        return x * self.fc(y).view(b, c, 1, 1)


## TRANSFORMS


In [9]:

def get_train_transform_aug():
    return T.Compose([
        T.RandomHorizontalFlip(p=0.5),
        T.RandomRotation(degrees=15),
        T.RandomAffine(degrees=0, translate=(0.10, 0.10),
                       scale=(0.90, 1.10), shear=3),
        T.ColorJitter(brightness=0.25, contrast=0.25,
                      saturation=0.15, hue=0.03),
        T.RandomAutocontrast(p=0.3),
        T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
        T.RandomAdjustSharpness(sharpness_factor=2, p=0.3),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


def get_val_transform():
    return T.Compose([
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


def get_noaug_transform():
    return T.Compose([
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


def resize_with_padding(img: Image.Image,
                        target=(256, 256)) -> Image.Image:
    w, h   = img.size
    ratio  = min(target[0] / w, target[1] / h)
    nw, nh = int(w * ratio), int(h * ratio)
    img    = img.resize((nw, nh), Image.BILINEAR)
    canvas = Image.new("RGB", target, (0, 0, 0))
    canvas.paste(img, ((target[0] - nw) // 2, (target[1] - nh) // 2))
    return canvas


## DATASET


In [10]:

def _autocontrast_gray(img: Image.Image) -> Image.Image:
    """Grayscale + autocontrast preprocessing before augmentation."""
    img = img.convert('L')
    img = ImageOps.autocontrast(img, cutoff=1)
    img = img.convert('RGB')
    return img


class ROIDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = str(row['roi_filepath'])

        img   = Image.open(path).convert("RGB")
        img   = _autocontrast_gray(img)
        img   = resize_with_padding(img, (cfg.IMAGE_SIZE, cfg.IMAGE_SIZE))

        label = cfg.CLASS_TO_LABEL.get(str(row['class']).lower(), -1)
        if label == -1:
            raise ValueError(
                f"Unknown label '{row['class']}' at index {idx}, path={path}.")

        if self.transform:
            img = self.transform(img)

        src = normalize_img_key(str(row['source_image']))
        return img, torch.tensor(label, dtype=torch.long), src


## LOSS


In [11]:

def build_criterion() -> nn.CrossEntropyLoss:
    """
    CrossEntropyLoss with label smoothing (cfg.LABEL_SMOOTHING = 0.1).
    Label smoothing prevents the model from driving logits to ±∞ to achieve
    zero loss — the primary mechanism causing the 99-100% train accuracy /
    rising val loss pattern seen in early runs.
    WeightedRandomSampler already delivers ~50/50 batches so no class weights.
    """
    return nn.CrossEntropyLoss(label_smoothing=cfg.LABEL_SMOOTHING)


## TEMPERATURE SCALING


In [12]:

class TemperatureScaler(nn.Module):
    """
    FIX (CRITICAL): Use log-parametrization so T = exp(log_T) is ALWAYS > 0.
    The original code optimized `self.temperature` directly; LBFGS can make
    large steps that drive the raw parameter deeply negative (observed: T=-11.24
    in Fold 4), inverting all predictions and collapsing AUC to ~0.06.
    With log-parametrization the parameter is unconstrained but T stays positive.
    """
    def __init__(self):
        super().__init__()
        # Initialise log_temp = log(1.5) ~ 0.405  ->  T_init = 1.5
        self.log_temp = nn.Parameter(torch.log(torch.ones(1) * 1.5))

    @property
    def temperature(self) -> torch.Tensor:
        return torch.exp(self.log_temp)

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        return logits / torch.exp(self.log_temp)


def fit_temperature(model: nn.Module,
                    loader: DataLoader,
                    device: torch.device,
                    n_iter: int = 200,
                    lr: float = 0.01) -> float:
    """
    FIX 1: Optimise log_temp (not temperature directly) -- T is always positive.
    FIX 2: Post-fit sanity clamp [0.1, 10.0] with warning if violated.
    """
    model.eval()
    scaler = TemperatureScaler().to(device)
    # FIX: optimise log_temp, not temperature
    opt    = optim.LBFGS([scaler.log_temp], lr=lr, max_iter=n_iter)
    nll    = nn.CrossEntropyLoss()

    all_logits, all_labels = [], []
    with torch.no_grad():
        for imgs, lbls, _ in loader:
            logits = model(imgs.to(device))
            all_logits.append(logits)
            all_labels.append(lbls.to(device))
    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)

    def closure():
        opt.zero_grad()
        loss = nll(scaler(all_logits), all_labels)
        loss.backward()
        return loss

    opt.step(closure)

    T = float(torch.exp(scaler.log_temp).item())

    # Safety clamp (belt-and-suspenders -- should never trigger with log-param)
    if not (0.1 <= T <= 10.0):
        print(f"     Warning: Temperature {T:.4f} outside [0.1, 10.0] -- clamping to 1.0")
        T = 1.0

    print(f"     Temperature scaling: T={T:.4f}")
    return T


# NOTE: Temperature is applied directly on logits (before softmax) in run_cv:
#   F.softmax(logits / T, dim=1)
# This is the correct and numerically stable approach.
# A post-hoc numpy version on probabilities is NOT used to avoid log-softmax
# numerical issues and is intentionally omitted.


# ============================================================================
# ROI → IMAGE-LEVEL AGGREGATION
#
# Multiple ROIs from the same source X-ray image are pooled to produce a
# single IMAGE-level prediction.
#
# Label  : majority vote across ROI labels; ties → malignant (safer).
#          DESIGN CHOICE — if an authoritative image-level label is
#          available in the manifest, that should be used instead.
# Prob   : max-pool on P(malignant) — one highly suspicious ROI is
#          sufficient to flag the whole image as malignant, consistent
#          with clinical "worst-region" reading practice.
#
# This produces IMAGE-level predictions, not patient-level predictions.
# Patient-level evaluation would require a further aggregation step
# grouping images by patient_id.
# ============================================================================

def aggregate_rois(labels, probs, img_ids):
    """
    Aggregate ROI-level predictions to IMAGE level.

    Returns:
        img_labels : (M,) int — image-level majority-vote labels
        img_probs  : (M, 2) float — image-level max-pooled probabilities
    """
    gp, gl = defaultdict(list), defaultdict(list)
    for y, p, i in zip(labels, probs, img_ids):
        k = normalize_img_key(i)
        gp[k].append(p)
        gl[k].append(int(y))

    img_labels, img_probs = [], []
    for k in sorted(gp):
        ll = gl[k]
        pp = np.stack(gp[k])   # (n_rois, 2)

        # Majority-vote label; ties → malignant (design choice — see docstring)
        cnt = Counter(ll)
        lbl = 1 if cnt[1] >= cnt[0] else cnt.most_common(1)[0][0]
        img_labels.append(lbl)

        # Max-pool on P(malignant)
        max_mal_idx = int(pp[:, 1].argmax())
        img_probs.append(pp[max_mal_idx])

    return np.array(img_labels), np.array(img_probs)


## THRESHOLD OPTIMISATION


In [13]:

def find_optimal_threshold(y_true: np.ndarray,
                            y_prob_mal: np.ndarray,
                            criterion: str = 'youden') -> float:
    """
    FIX: Guard against degenerate / inverted models.

    sklearn's roc_curve() inserts an implicit first threshold of
    max(y_score)+1 (point where nothing is predicted positive).  When the
    model is inverted (AUC < 0.5) every youden = tpr-fpr value is ≤ 0, so
    argmax picks index 0 → threshold = max_score+1 which rounds to inf after
    the print format.  np.clip(inf, 0.05, 0.95) = 0.95, giving Sens=0.

    Fixes applied:
      1. Filter the sentinel "no-positive" threshold before computing youden.
      2. Detect AUC < 0.5 (inverted model) and fall back to 0.5 with warning.
      3. Ensure best_thr is always a finite float.
    """
    auc = float(roc_auc_score(y_true, y_prob_mal)) if len(np.unique(y_true)) > 1 else 0.5

    if auc < 0.5:
        print(f"     ⚠️  AUC-ROC={auc:.3f} < 0.5 — model predictions may be "
              f"inverted (check temperature scaling). Falling back to thr=0.5.")
        return 0.5

    fpr, tpr, thresholds = roc_curve(y_true, y_prob_mal)

    # FIX: Drop the sentinel inf/out-of-range first threshold that sklearn adds.
    # It represents "predict nothing positive" and should not be a candidate.
    finite_mask = np.isfinite(thresholds) & (thresholds <= 1.0 + 1e-6)
    if finite_mask.sum() < 2:
        print("     ⚠️  Too few finite thresholds — falling back to thr=0.5.")
        return 0.5
    fpr, tpr, thresholds = fpr[finite_mask], tpr[finite_mask], thresholds[finite_mask]

    if criterion == 'youden':
        youden   = tpr - fpr
        best_idx = int(np.argmax(youden))
        best_thr = float(thresholds[best_idx])
        print(f"     🎯 Optimal threshold (Youden): {best_thr:.4f}  "
              f"(Sens={tpr[best_idx]:.3f}, Spec={1-fpr[best_idx]:.3f})")

    elif criterion == 'f1':
        best_thr, best_f1 = 0.5, -1.0
        for thr in thresholds:
            preds = (y_prob_mal >= thr).astype(int)
            _, _, f1, _ = precision_recall_fscore_support(
                y_true, preds, labels=[0, 1], zero_division=0)
            if f1[1] > best_f1:
                best_f1  = f1[1]
                best_thr = float(thr)
        print(f"     🎯 Optimal threshold (F1-mal): {best_thr:.4f}  "
              f"(F1={best_f1:.3f})")
    else:
        raise ValueError(f"Unknown threshold criterion: {criterion}")

    # FIX: ensure finite result before clipping
    if not np.isfinite(best_thr):
        print(f"     ⚠️  Non-finite threshold {best_thr} — falling back to 0.5.")
        best_thr = 0.5

    return float(np.clip(best_thr, 0.05, 0.95))


# ============================================================================
# METRICS
# FIX — ECE: last bin now includes probability == 1.0 (right-edge closed)
# ============================================================================

def ece(y_true, y_prob, n_bins=10):
    """
    Expected Calibration Error.
    Bins are half-open [lo, hi) except the last which is [lo, 1.0] so that
    predictions of exactly 1.0 are not silently excluded.
    """
    bins = np.linspace(0, 1, n_bins + 1)
    e = 0.0
    n = len(y_true)
    for i, (lo, hi) in enumerate(zip(bins[:-1], bins[1:])):
        if i < n_bins - 1:
            m = (y_prob >= lo) & (y_prob < hi)   # half-open: [lo, hi)
        else:
            m = (y_prob >= lo) & (y_prob <= hi)  # FIX: last bin is closed [lo, 1.0]
        if m.sum():
            e += m.sum() / n * abs(y_true[m].mean() - y_prob[m].mean())
    return float(e)


_SCALAR_KEYS = [
    'accuracy', 'balanced_accuracy', 'f1_macro',
    'auc_roc', 'auc_pr', 'mcc', 'brier_score', 'ece',
    'sensitivity', 'specificity',
    'benign_precision', 'benign_recall', 'benign_f1',
    'malignant_precision', 'malignant_recall', 'malignant_f1',
    'optimal_threshold',
]


def compute_metrics(y_true, y_pred, y_probs):
    pr, rc, f1, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1], zero_division=0)

    acc  = float((y_pred == y_true).mean())
    bacc = float((rc[0] + rc[1]) / 2)
    f1m  = float(f1.mean())

    try:    auc_r = float(roc_auc_score(y_true, y_probs[:, 1]))
    except: auc_r = 0.0
    try:    auc_p = float(average_precision_score(y_true, y_probs[:, 1]))
    except: auc_p = 0.0
    try:    mcc_v = float(matthews_corrcoef(y_true, y_pred))
    except: mcc_v = 0.0
    try:    brier = float(brier_score_loss(y_true, y_probs[:, 1]))
    except: brier = 1.0

    ece_v = ece(y_true, y_probs[:, 1], cfg.CALIB_N_BINS)

    cm             = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = float(tp / (tp + fn)) if (tp + fn) else 0.0
    spec = float(tn / (tn + fp)) if (tn + fp) else 0.0

    try:
        fpr_a, tpr_a, _ = roc_curve(y_true, y_probs[:, 1])
        p_a, r_a, _     = precision_recall_curve(y_true, y_probs[:, 1])
    except:
        fpr_a = tpr_a = p_a = r_a = np.array([0., 1.])

    clf_rep = classification_report(
        y_true, y_pred, target_names=cfg.CLASS_NAMES,
        output_dict=True, zero_division=0)

    return {
        'accuracy':            acc,   'balanced_accuracy': bacc,
        'f1_macro':            f1m,   'auc_roc':           auc_r,
        'auc_pr':              auc_p, 'mcc':               mcc_v,
        'brier_score':         brier, 'ece':               ece_v,
        'sensitivity':         sens,  'specificity':       spec,
        'benign_precision':    float(pr[0]), 'benign_recall':    float(rc[0]),
        'benign_f1':           float(f1[0]),
        'malignant_precision': float(pr[1]), 'malignant_recall': float(rc[1]),
        'malignant_f1':        float(f1[1]),
        'benign_support':      int(sup[0]),  'malignant_support': int(sup[1]),
        'optimal_threshold':   0.5,   # placeholder; overwritten by caller
        'fpr':             fpr_a, 'tpr':            tpr_a,
        'precision_curve': p_a,   'recall_curve':   r_a,
        'confusion_matrix':    cm,
        'classification_report': clf_rep,
    }


def aggregate_cv_metrics(fold_metrics):
    """
    FIX: Detect and exclude collapsed folds (AUC-ROC < 0.5) before averaging.

    A fold is considered "collapsed" when its AUC-ROC < 0.5, which means the
    model's predictions are worse than random chance — typically caused by
    an inverted temperature (now fixed at source) or a degenerate fold split.

    Collapsed folds are:
      - Reported in the summary so they are not silently hidden.
      - Excluded from the mean/std so a single bad fold does not skew results.
      - Their raw per-fold values are still stored in *_folds for diagnostics.
    """
    auc_roc_vals = [m.get('auc_roc', 0.0) for m in fold_metrics]
    collapsed    = [i for i, v in enumerate(auc_roc_vals) if v < 0.5]
    healthy      = [m for i, m in enumerate(fold_metrics) if i not in collapsed]

    if collapsed:
        print(f"  ⚠️  Collapsed folds detected (AUC-ROC < 0.5): "
              f"{[i+1 for i in collapsed]}  — excluded from mean/std.")
    if not healthy:
        print("  ❌ ALL folds collapsed — returning raw aggregate (all folds).")
        healthy = fold_metrics  # last-resort: return something

    agg = {}
    agg['n_folds_total']    = len(fold_metrics)
    agg['n_folds_excluded'] = len(collapsed)
    agg['excluded_fold_ids'] = [i + 1 for i in collapsed]

    for k in _SCALAR_KEYS:
        all_vals     = [m[k] for m in fold_metrics if k in m]
        healthy_vals = [m[k] for m in healthy       if k in m]
        agg[f'{k}_mean']  = float(np.mean(healthy_vals)) if healthy_vals else 0.0
        agg[f'{k}_std']   = float(np.std(healthy_vals))  if healthy_vals else 0.0
        agg[f'{k}_folds'] = [float(v) for v in all_vals]   # all folds for violin plots
    return agg


# ============================================================================
# MIXUP AUGMENTATION
# Only used for the 'with_aug' CV condition.
# Interpolates pairs of training images and their labels using Beta(alpha, alpha)
# mixing coefficient.  Forces the model to behave linearly between training
# examples, which prevents sharp memorisation of individual samples.
# ============================================================================

def mixup_data(imgs: torch.Tensor, lbls: torch.Tensor,
               alpha: float = 0.4) -> tuple:
    """
    Returns mixed inputs, pairs of labels, and mixing coefficient lambda.
    lam ~ Beta(alpha, alpha); if alpha<=0 returns originals unchanged.
    """
    if alpha <= 0.0:
        return imgs, lbls, lbls, 1.0

    lam = float(np.random.beta(alpha, alpha))
    # Ensure the dominant image always has lam >= 0.5 for stability
    lam = max(lam, 1.0 - lam)

    batch_size = imgs.size(0)
    idx = torch.randperm(batch_size, device=imgs.device)
    mixed_imgs = lam * imgs + (1.0 - lam) * imgs[idx]
    return mixed_imgs, lbls, lbls[idx], lam


def mixup_criterion(criterion: nn.Module,
                    pred: torch.Tensor,
                    lbl_a: torch.Tensor,
                    lbl_b: torch.Tensor,
                    lam: float) -> torch.Tensor:
    """Weighted sum of CE losses for the two mixed label sets."""
    return lam * criterion(pred, lbl_a) + (1.0 - lam) * criterion(pred, lbl_b)


## ONE EPOCH — train


In [14]:

def run_train_epoch(model, loader, criterion, optimizer, device, accum_steps,
                    use_mixup: bool = False):
    model.train()
    tot_loss, correct, total, accum = 0., 0, 0, 0
    optimizer.zero_grad()

    for imgs, lbls, _ in loader:
        imgs = imgs.to(device); lbls = lbls.to(device)

        if use_mixup:
            imgs, lbl_a, lbl_b, lam = mixup_data(imgs, lbls, cfg.MIXUP_ALPHA)
            out  = model(imgs)
            loss = mixup_criterion(criterion, out, lbl_a, lbl_b, lam) / accum_steps
        else:
            out  = model(imgs)
            loss = criterion(out, lbls) / accum_steps

        loss.backward(); accum += 1

        if accum % accum_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            optimizer.step(); optimizer.zero_grad()

        tot_loss += loss.item() * accum_steps
        correct  += (out.argmax(1) == lbls).sum().item()
        total    += lbls.size(0)

    if accum % accum_steps != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
        optimizer.step(); optimizer.zero_grad()

    return tot_loss / len(loader), correct / total


## ONE EPOCH — evaluate (IMAGE-level aggregation)


In [15]:

def run_eval_epoch(model, loader, criterion, device, threshold: float = 0.5):
    """
    Evaluate model on a loader.
    Predictions are aggregated to IMAGE level via aggregate_rois before
    metric computation.  Threshold controls the malignant decision boundary.
    """
    model.eval()
    tot_loss, all_lbl, all_prob, all_ids = 0., [], [], []

    with torch.no_grad():
        for imgs, lbls, ids in loader:
            imgs = imgs.to(device); lbls = lbls.to(device)
            out  = model(imgs)
            tot_loss += criterion(out, lbls).item()
            all_lbl.extend(lbls.cpu().numpy())
            all_prob.extend(F.softmax(out, 1).cpu().numpy())
            all_ids.extend(ids)

    all_lbl  = np.array(all_lbl)
    all_prob = np.array(all_prob)

    # IMAGE-level aggregation (max-pool on P(malignant))
    img_lbl, img_prob = aggregate_rois(all_lbl, all_prob, all_ids)
    img_pred = (img_prob[:, 1] >= threshold).astype(int)

    m = compute_metrics(img_lbl, img_pred, img_prob)
    return tot_loss / len(loader), m


## TRAINING LOOP


In [16]:

def train_loop(model, tr_loader, va_loader, device,
               n_epochs, save_path, verbose=True, aug_cond: str = 'no_aug'):
    """
    aug_cond : 'with_aug' enables MixUp; 'no_aug' disables it.
    """
    criterion  = build_criterion()
    use_mixup  = (aug_cond == 'with_aug')

    freeze_backbone(model)
    opt = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg.LEARNING_RATE, weight_decay=cfg.WEIGHT_DECAY)

    wsched = WarmupScheduler(opt, cfg.WARMUP_EPOCHS,
                             cfg.LEARNING_RATE) if cfg.USE_WARMUP else None
    psched = optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='max', factor=0.5, patience=4, min_lr=1e-7)
    ema    = EMAMetric(cfg.EMA_ALPHA)

    best   = -float('inf'); pat = 0; unfrozen = False
    hist   = {'train_loss': [], 'train_acc': [], 'val_loss': [],
              'val_loss_ema': [], 'val_metrics': []}

    for ep in range(n_epochs):
        if wsched and wsched.is_warmup():
            wsched.step()

        if ep == cfg.UNFREEZE_EPOCH and not unfrozen:
            unfreeze_backbone(model)
            # Keep BatchNorm layers in eval() and frozen — prevents BN running
            # stats from over-adapting to the small training split.
            freeze_bn_layers(model)
            unfrozen = True
            opt = optim.AdamW(model.parameters(),
                              lr=cfg.FINETUNE_LR,
                              weight_decay=cfg.WEIGHT_DECAY)
            wsched = WarmupScheduler(opt, cfg.WARMUP_EPOCHS,
                                     cfg.FINETUNE_LR) if cfg.USE_WARMUP else None
            psched = optim.lr_scheduler.ReduceLROnPlateau(
                opt, mode='max', factor=0.5, patience=3, min_lr=1e-7)
            if verbose: print(f"     🔓 Unfreeze @ ep {ep+1} (BN layers kept frozen)")

        tl, ta = run_train_epoch(model, tr_loader, criterion, opt,
                                  device, cfg.GRADIENT_ACCUMULATION_STEPS,
                                  use_mixup=use_mixup)
        vl, vm = run_eval_epoch(model, va_loader, criterion, device,
                                threshold=0.5)
        ve  = ema.update(vl)
        cur = vm[cfg.MONITOR_METRIC]

        if not (wsched and wsched.is_warmup()):
            psched.step(cur)

        hist['train_loss'].append(float(tl))
        hist['train_acc'].append(float(ta))
        hist['val_loss'].append(float(vl))
        hist['val_loss_ema'].append(float(ve))
        hist['val_metrics'].append(vm)

        if cur > best + cfg.MIN_DELTA:
            best = cur; pat = 0
            torch.save({'epoch': ep + 1, 'model_state_dict': model.state_dict(),
                        'best_metric': best}, save_path)
        else:
            if ep >= cfg.MIN_EPOCHS: pat += 1

        if verbose and (ep + 1) % 5 == 0:
            print(f"     Ep{ep+1:3d} | TL={tl:.4f} TA={ta*100:.1f}% | "
                  f"VL={vl:.4f} AUC-PR={cur:.4f} | "
                  f"Pat={pat}/{cfg.EARLY_STOP_PATIENCE}")

        if pat >= cfg.EARLY_STOP_PATIENCE and ep >= cfg.MIN_EPOCHS:
            if verbose: print(f"     ⛔ Early stop ep{ep+1} | best={best:.4f}")
            break

    return hist


## PATIENT GROUPS LOADER


In [17]:

def load_metadata_patient_groups(df: pd.DataFrame) -> np.ndarray:
    """
    Return per-ROI group array (patient_id) for StratifiedGroupKFold.
    patient_id is read from the DataFrame (loaded from METADATA_CSV).
    """
    if 'patient_id' not in df.columns:
        print("  ⚠️  'patient_id' not found — falling back to source_image stem.")
        return _fallback_groups(df)

    groups     = df['patient_id'].astype(str).values
    n_patients = len(set(groups))
    n_unknown  = int(np.isin(groups, ['unknown', 'nan']).sum())

    print(f"  Patient groups : {n_patients} unique patients → {len(df)} ROI rows")
    print(f"  ROIs / patient : {len(df) / max(n_patients, 1):.1f} avg")

    if n_unknown > 0:
        print(f"  ⚠️  {n_unknown} ROIs have missing patient_id.")

    min_per_fold = n_patients // cfg.N_FOLDS
    if min_per_fold < 5:
        print(f"  ⚠️  Only ~{min_per_fold} patients per fold — "
              f"consider reducing N_FOLDS.")

    return groups


def _fallback_groups(df: pd.DataFrame) -> np.ndarray:
    groups   = np.array([Path(str(r['source_image'])).stem
                         for _, r in df.iterrows()])
    n_groups = len(set(groups))
    print(f"  Fallback grouping: {n_groups} unique source images → {len(df)} ROIs")
    return groups


## MODELS


In [18]:

class TumorClassifierDenseNet121SE(nn.Module):
    def __init__(self, n=2, r=16):
        super().__init__()
        bb = torchvision.models.densenet121(weights='IMAGENET1K_V1')
        self.features = bb.features
        self.se       = SEBlock(1024, r)
        self.pool     = nn.AdaptiveAvgPool2d(1)
        self.head     = nn.Sequential(
            nn.Dropout(cfg.HEAD_DROPOUT_1), nn.Linear(1024, 256), nn.ReLU(),
            nn.Dropout(cfg.HEAD_DROPOUT_2), nn.Linear(256, n))
    def forward(self, x):
        x = self.pool(self.se(self.features(x)))
        return self.head(torch.flatten(x, 1))


class TumorClassifierResNet18SE(nn.Module):
    def __init__(self, n=2, r=16):
        super().__init__()
        bb = torchvision.models.resnet18(weights='IMAGENET1K_V1')
        self.stem = nn.Sequential(bb.conv1, bb.bn1, bb.relu, bb.maxpool)
        self.l1, self.l2, self.l3, self.l4 = (
            bb.layer1, bb.layer2, bb.layer3, bb.layer4)
        self.se1 = SEBlock(64, r);  self.se2 = SEBlock(128, r)
        self.se3 = SEBlock(256, r); self.se4 = SEBlock(512, r)
        self.pool = bb.avgpool
        self.head = nn.Sequential(
            nn.Dropout(cfg.HEAD_DROPOUT_1), nn.Linear(512, 256), nn.ReLU(),
            nn.Dropout(cfg.HEAD_DROPOUT_2), nn.Linear(256, n))
    def forward(self, x):
        x = self.stem(x)
        x = self.se1(self.l1(x)); x = self.se2(self.l2(x))
        x = self.se3(self.l3(x)); x = self.se4(self.l4(x))
        return self.head(torch.flatten(self.pool(x), 1))


class TumorClassifierEfficientNetB0SE(nn.Module):
    def __init__(self, n=2, r=16):
        super().__init__()
        bb = torchvision.models.efficientnet_b0(weights='IMAGENET1K_V1')
        self.features = bb.features; self.pool = bb.avgpool
        self.se       = SEBlock(1280, r)
        self.head     = nn.Sequential(
            nn.Dropout(cfg.HEAD_DROPOUT_1), nn.Linear(1280, 256), nn.ReLU(),
            nn.Dropout(cfg.HEAD_DROPOUT_2), nn.Linear(256, n))
    def forward(self, x):
        x = self.pool(self.se(self.features(x)))
        return self.head(torch.flatten(x, 1))


class TumorClassifierMobileNetV2SE(nn.Module):
    def __init__(self, n=2, r=16):
        super().__init__()
        bb = torchvision.models.mobilenet_v2(weights='IMAGENET1K_V1')
        self.features = bb.features; self.pool = nn.AdaptiveAvgPool2d(1)
        self.se       = SEBlock(1280, r)
        self.head     = nn.Sequential(
            nn.Dropout(cfg.HEAD_DROPOUT_1), nn.Linear(1280, 256), nn.ReLU(),
            nn.Dropout(cfg.HEAD_DROPOUT_2), nn.Linear(256, n))
    def forward(self, x):
        x = self.pool(self.se(self.features(x)))
        return self.head(torch.flatten(x, 1))


class TumorClassifierConvNeXtTinySE(nn.Module):
    def __init__(self, n=2, r=16):
        super().__init__()
        bb = torchvision.models.convnext_tiny(weights='IMAGENET1K_V1')
        self.features = bb.features
        self.se       = SEBlock(768, r)
        self.pool     = nn.AdaptiveAvgPool2d(1)
        self.head     = nn.Sequential(
            nn.Flatten(), nn.LayerNorm(768),
            nn.Dropout(cfg.HEAD_DROPOUT_1), nn.Linear(768, 256), nn.GELU(),
            nn.Dropout(cfg.HEAD_DROPOUT_2), nn.Linear(256, n))
    def forward(self, x):
        x = self.pool(self.se(self.features(x)))
        return self.head(x)


class TumorClassifierEfficientNetB2SE(nn.Module):
    def __init__(self, n=2, r=16):
        super().__init__()
        bb = torchvision.models.efficientnet_b2(weights='IMAGENET1K_V1')
        self.features = bb.features; self.pool = bb.avgpool
        self.se       = SEBlock(1408, r)
        self.head     = nn.Sequential(
            nn.Dropout(cfg.HEAD_DROPOUT_1), nn.Linear(1408, 256), nn.SiLU(),
            nn.Dropout(cfg.HEAD_DROPOUT_2), nn.Linear(256, n))
    def forward(self, x):
        x = self.pool(self.se(self.features(x)))
        return self.head(torch.flatten(x, 1))


_REGISTRY = {
    'densenet121_se':     (TumorClassifierDenseNet121SE,  {}),
    'resnet18_se':        (TumorClassifierResNet18SE,     {}),
    'efficientnet_b0_se': (TumorClassifierEfficientNetB0SE, {}),
    'mobilenet_v2_se':    (TumorClassifierMobileNetV2SE,  {}),
    'convnext_tiny_se':   (TumorClassifierConvNeXtTinySE, {}),
    'efficientnet_b2_se': (TumorClassifierEfficientNetB2SE, {}),
}


def get_model(name: str) -> nn.Module:
    cls, kw = _REGISTRY[name]
    m   = cls(n=cfg.NUM_CLASSES, r=cfg.SE_REDUCTION, **kw)
    tot = sum(p.numel() for p in m.parameters())
    trn = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"  ✅ {name}: {tot/1e6:.1f}M total / {trn/1e6:.1f}M trainable")
    return m


## FREEZE / UNFREEZE


In [19]:

def freeze_backbone(model: nn.Module):
    for nm, p in model.named_parameters():
        if not any(k in nm for k in ('se', 'head', 'classifier', 'fc')):
            p.requires_grad = False


def unfreeze_backbone(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = True


def freeze_bn_layers(model: nn.Module):
    """
    After unfreezing the backbone, keep all BatchNorm layers in eval() mode
    and freeze their weight/bias.

    Why: with small fold sizes (~1400 ROIs), the running mean/variance of BN
    layers will quickly adapt to training-set statistics and overfit.
    Freezing BN forces the model to use the ImageNet-pretrained statistics,
    which generalise better on small datasets.

    This is a standard practice for fine-tuning on medical imaging datasets.
    """
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            m.eval()                   # keeps running_mean/var fixed
            for p in m.parameters():
                p.requires_grad = False   # freezes gamma and beta


## WARMUP SCHEDULER


In [20]:

class WarmupScheduler:
    def __init__(self, opt, warmup_epochs=5, base_lr=None):
        self.opt    = opt
        self.epochs = warmup_epochs
        self.base   = base_lr or opt.param_groups[0]['lr']
        self.local  = 0

    def step(self):
        if self.local < self.epochs:
            lr = self.base * (self.local + 1) / self.epochs
            for pg in self.opt.param_groups: pg['lr'] = lr
        self.local += 1

    def is_warmup(self): return self.local < self.epochs


class EMAMetric:
    def __init__(self, a=0.3): self.a = a; self.v = None
    def update(self, x):
        self.v = x if self.v is None else self.a * x + (1 - self.a) * self.v
        return self.v


## DATALOADER FACTORY


In [21]:

def _seed_worker(wid):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)


def make_loader(df: pd.DataFrame, transform,
                shuffle=False, weighted=False) -> DataLoader:
    ds = ROIDataset(df, transform)
    g  = torch.Generator(); g.manual_seed(cfg.RANDOM_SEED)

    if weighted and shuffle:
        labels = [cfg.CLASS_TO_LABEL.get(str(r['class']).lower(), 0)
                  for _, r in df.iterrows()]
        cnts   = Counter(labels)
        sw     = {0: 1.0 / cnts[0], 1: 1.0 / cnts[1]}
        sw_arr = [sw[l] for l in labels]
        total_w = sum(sw_arr)
        mal_frac = sum(sw_arr[i] for i, l in enumerate(labels) if l == 1) / total_w
        print(f"     Sampler: benign≈{(1-mal_frac)*100:.1f}%  "
              f"malignant≈{mal_frac*100:.1f}%  (target 50/50)")
        sampler = WeightedRandomSampler(sw_arr, len(sw_arr), replacement=True)
        return DataLoader(ds, batch_size=cfg.BATCH_SIZE, sampler=sampler,
                          num_workers=cfg.NUM_WORKERS, pin_memory=True,
                          worker_init_fn=_seed_worker, generator=g)

    return DataLoader(ds, batch_size=cfg.BATCH_SIZE, shuffle=shuffle,
                      num_workers=cfg.NUM_WORKERS, pin_memory=True,
                      worker_init_fn=_seed_worker, generator=g)


## PLOTTING


In [22]:

COLORS = plt.cm.tab10.colors


def _save(fig, path):
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  📊 Saved: {os.path.basename(path)}")


def plot_imbalance_overview(stats_dict: dict, save_dir: str):
    splits = list(stats_dict.keys())
    bens   = [stats_dict[s].get('n_benign',    0) for s in splits]
    mals   = [stats_dict[s].get('n_malignant', 0) for s in splits]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    x = np.arange(len(splits))
    axes[0].bar(x, bens, label='Benign',    color='#3498DB', alpha=0.85)
    axes[0].bar(x, mals, label='Malignant', color='#E74C3C', alpha=0.85,
                bottom=bens)
    axes[0].set_xticks(x); axes[0].set_xticklabels(splits)
    axes[0].set_title('Class Distribution per Split')
    axes[0].set_ylabel('ROI Count'); axes[0].legend()

    ratios = [stats_dict[s].get('ratio', 1) for s in splits]
    axes[1].bar(splits, ratios, color='#9B59B6', alpha=0.85)
    axes[1].axhline(1, color='grey', ls='--', lw=1)
    axes[1].set_title('Benign:Malignant Ratio per Split')
    axes[1].set_ylabel('Ratio (>1 = class imbalance)')
    for i, v in enumerate(ratios):
        axes[1].text(i, v + 0.05, f'{v:.2f}x', ha='center', fontsize=9)

    fig.suptitle('Class Imbalance Overview', fontsize=13, fontweight='bold')
    plt.tight_layout()
    _save(fig, os.path.join(save_dir, "class_imbalance_overview.png"))


def plot_roc_pr_curves(fold_metrics, save_dir, prefix=""):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    mean_fpr  = np.linspace(0, 1, 200)
    tprs, prec_interps, aps = [], [], []

    for k, m in enumerate(fold_metrics):
        fpr, tpr = np.array(m['fpr']), np.array(m['tpr'])
        prec     = np.array(m['precision_curve'])
        rec      = np.array(m['recall_curve'])
        thr      = m.get('optimal_threshold', 0.5)

        tpr_i = np.interp(mean_fpr, fpr, tpr); tpr_i[0] = 0.0
        tprs.append(tpr_i)
        axes[0].plot(fpr, tpr, lw=1, alpha=0.55, color=COLORS[k % 10],
                     label=f"F{k+1} AUC={m['auc_roc']:.3f} (thr={thr:.2f})")

        rec_g  = np.linspace(0, 1, 200)[::-1]
        prec_i = np.interp(rec_g, rec[::-1], prec[::-1])
        prec_interps.append(prec_i); aps.append(m['auc_pr'])
        axes[1].plot(rec, prec, lw=1, alpha=0.55, color=COLORS[k % 10],
                     label=f"F{k+1} AP={m['auc_pr']:.3f}")

    mt = np.mean(tprs, axis=0); st = np.std(tprs, axis=0); mt[-1] = 1.0
    axes[0].plot(mean_fpr, mt, 'k-', lw=2.5,
                 label=f"Mean AUC={np.mean([m['auc_roc'] for m in fold_metrics]):.3f}"
                       f"±{np.std([m['auc_roc'] for m in fold_metrics]):.3f}")
    axes[0].fill_between(mean_fpr, mt - st, mt + st, alpha=0.15, color='grey')
    axes[0].plot([0, 1], [0, 1], '--', color='silver', lw=1)
    axes[0].set(title=f'{prefix}\nROC Curve', xlabel='FPR', ylabel='TPR')
    axes[0].legend(fontsize=7.5); axes[0].grid(alpha=0.3)

    mp = np.mean(prec_interps, axis=0); sp = np.std(prec_interps, axis=0)
    rg = np.linspace(0, 1, 200)[::-1]
    axes[1].plot(rg, mp, 'k-', lw=2.5,
                 label=f"Mean AP={np.mean(aps):.3f}±{np.std(aps):.3f}")
    axes[1].fill_between(rg, mp - sp, mp + sp, alpha=0.15, color='grey')
    axes[1].set(title=f'{prefix}\nPR Curve', xlabel='Recall', ylabel='Precision')
    axes[1].legend(fontsize=7.5); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    _save(fig, os.path.join(save_dir, "roc_pr_curves.png"))


def plot_confusion_matrices(fold_metrics, save_dir, prefix=""):
    n    = len(fold_metrics)
    cols = min(n + 1, 4); rows = (n + 1 + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4.5 * cols, 4.2 * rows))
    axes   = np.array(axes).ravel()
    names  = cfg.CLASS_NAMES
    agg_cm = np.zeros((2, 2), dtype=int)

    for k, m in enumerate(fold_metrics):
        cm = np.array(m['confusion_matrix'])
        agg_cm += cm
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=names, yticklabels=names,
                    ax=axes[k], cbar=False)
        tn, fp, fn, tp = cm.ravel()
        thr = m.get('optimal_threshold', 0.5)
        axes[k].set_title(
            f"Fold {k+1} (thr={thr:.2f})\n"
            f"Sens={tp/(tp+fn):.2f}  Spec={tn/(tn+fp):.2f}", fontsize=9)
        axes[k].set_xlabel("Predicted"); axes[k].set_ylabel("True")

    sns.heatmap(agg_cm, annot=True, fmt='d', cmap='Oranges',
                xticklabels=names, yticklabels=names,
                ax=axes[n], cbar=False)
    tn, fp, fn, tp = agg_cm.ravel()
    axes[n].set_title(
        f"Aggregate\nSens={tp/(tp+fn):.2f}  Spec={tn/(tn+fp):.2f}",
        fontsize=9, fontweight='bold')
    axes[n].set_xlabel("Predicted"); axes[n].set_ylabel("True")
    for i in range(n + 1, len(axes)): axes[i].set_visible(False)

    fig.suptitle(f"{prefix} — Confusion Matrices (Image-Level)",
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    _save(fig, os.path.join(save_dir, "confusion_matrices.png"))


def plot_training_curves(history, save_dir, prefix=""):
    ep = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    axes[0].plot(ep, history['train_loss'], label='Train')
    axes[0].plot(ep, history['val_loss'],   label='Val', ls='--', alpha=0.6)
    if cfg.USE_EMA:
        axes[0].plot(ep, history['val_loss_ema'],
                     label=f'EMA(α={cfg.EMA_ALPHA})', lw=2)
    axes[0].set(title='Loss', xlabel='Epoch')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    auc_r = [m['auc_roc'] for m in history['val_metrics']]
    auc_p = [m['auc_pr']  for m in history['val_metrics']]
    axes[1].plot(ep, auc_r, label='AUC-ROC')
    axes[1].plot(ep, auc_p, label='AUC-PR ⭐', lw=2.5)
    axes[1].set(title='AUC (Image-Level Val)', xlabel='Epoch')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    sens  = [m['sensitivity'] for m in history['val_metrics']]
    spec  = [m['specificity'] for m in history['val_metrics']]
    mcc   = [m['mcc']         for m in history['val_metrics']]
    brier = [m['brier_score'] for m in history['val_metrics']]
    axes[2].plot(ep, sens,  label='Sensitivity')
    axes[2].plot(ep, spec,  label='Specificity')
    axes[2].plot(ep, mcc,   label='MCC',   ls=':')
    axes[2].plot(ep, brier, label='Brier', ls='--', alpha=0.7)
    axes[2].set(title='Imbalance-sensitive Metrics (Image-Level)', xlabel='Epoch')
    axes[2].legend(); axes[2].grid(alpha=0.3)

    fig.suptitle(f"{prefix} — Training Curves", fontsize=12, fontweight='bold')
    plt.tight_layout()
    _save(fig, os.path.join(save_dir, "training_curves.png"))


def plot_calibration(fold_metrics, save_dir, prefix=""):
    """
    Calibration curve plotted on ROI-level probabilities (pre-aggregation).
    Labelled explicitly as ROI-level to distinguish from image-level metrics.
    For image-level calibration, use the post-aggregation probabilities
    stored in '_img_prob_raw'.
    """
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax in axes:
        ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Perfect calibration')

    ece_roi_vals = [m['ece'] for m in fold_metrics]
    ece_img_vals = [m.get('ece_image_level', m['ece']) for m in fold_metrics]

    for k, m in enumerate(fold_metrics):
        T   = m.get('temperature', 1.0)
        thr = m.get('optimal_threshold', 0.5)

        # ── ROI-level calibration ─────────────────────────────────────────
        yt_roi = m.get('_y_true_raw')
        yp_roi = m.get('_y_prob_raw')
        if yt_roi is not None:
            pt, pp = calibration_curve(yt_roi, yp_roi,
                                        n_bins=cfg.CALIB_N_BINS,
                                        strategy='uniform')
            axes[0].plot(pp, pt, 'o-', lw=1.5, alpha=0.7,
                         color=COLORS[k % 10],
                         label=f"Fold {k+1} (ECE={m['ece']:.3f}, T={T:.2f})")

        # ── Image-level calibration ───────────────────────────────────────
        yt_img = m.get('_img_lbl_raw')
        yp_img = m.get('_img_prob_raw')
        if yt_img is not None:
            ece_img = ece(yt_img, yp_img, cfg.CALIB_N_BINS)
            m['ece_image_level'] = ece_img
            pt2, pp2 = calibration_curve(yt_img, yp_img,
                                          n_bins=cfg.CALIB_N_BINS,
                                          strategy='uniform')
            axes[1].plot(pp2, pt2, 'o-', lw=1.5, alpha=0.7,
                         color=COLORS[k % 10],
                         label=f"Fold {k+1} (ECE={ece_img:.3f}, T={T:.2f},"
                               f" thr={thr:.2f})")

    axes[0].set(title=f"{prefix}\nROI-Level Calibration — "
                      f"Mean ECE={np.mean(ece_roi_vals):.3f}"
                      f"±{np.std(ece_roi_vals):.3f}",
                xlabel='Mean Predicted Prob', ylabel='Fraction of Positives')
    axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

    axes[1].set(title=f"{prefix}\nImage-Level Calibration — "
                      f"Mean ECE={np.mean(ece_img_vals):.3f}"
                      f"±{np.std(ece_img_vals):.3f}",
                xlabel='Mean Predicted Prob', ylabel='Fraction of Positives')
    axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    _save(fig, os.path.join(save_dir, "calibration_curve.png"))


def plot_cross_model_comparison(summary_df, save_dir):
    cols   = ['auc_roc_mean', 'auc_pr_mean', 'f1_macro_mean',
               'sensitivity_mean', 'specificity_mean', 'mcc_mean']
    labels = ['AUC-ROC', 'AUC-PR', 'F1-Macro',
              'Sensitivity', 'Specificity', 'MCC']

    models = summary_df['model'].unique()
    conds  = summary_df['aug_condition'].unique()
    x      = np.arange(len(models)); w = 0.35

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Cross-Model CV Comparison (Image-Level) — Mean ± Std',
                 fontsize=14, fontweight='bold')
    axes = axes.ravel()

    for ai, (col, lab) in enumerate(zip(cols, labels)):
        std_col = col.replace('_mean', '_std')
        ax = axes[ai]
        for j, cond in enumerate(conds):
            sub = summary_df[summary_df['aug_condition'] == cond].set_index('model')
            ms  = [sub.loc[m, col]     if m in sub.index else 0 for m in models]
            ss  = [sub.loc[m, std_col] if m in sub.index else 0 for m in models]
            ax.bar(x + j * w - w / 2, ms, w, yerr=ss, capsize=3,
                   label=cond, alpha=0.85, color=COLORS[j % 10])
        ax.set_title(lab, fontsize=11)
        ax.set_xticks(x)
        ax.set_xticklabels([m.replace('_se', '') for m in models],
                            rotation=25, ha='right', fontsize=8)
        ax.set_ylim(0, 1.05); ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    _save(fig, os.path.join(save_dir, "cross_model_comparison.png"))


def plot_aug_vs_noaug(summary_df, save_dir):
    cols   = ['auc_roc_mean', 'auc_pr_mean', 'f1_macro_mean',
               'sensitivity_mean', 'mcc_mean']
    labels = ['AUC-ROC', 'AUC-PR', 'F1-Macro', 'Sensitivity', 'MCC']

    aug_df   = summary_df[summary_df['aug_condition'] == 'with_aug'].set_index('model')
    noaug_df = summary_df[summary_df['aug_condition'] == 'no_aug'].set_index('model')
    models   = aug_df.index.intersection(noaug_df.index)

    fig, axes = plt.subplots(1, len(cols), figsize=(4 * len(cols), 5))
    for ax, col, lab in zip(axes, cols, labels):
        deltas = [aug_df.loc[m, col] - noaug_df.loc[m, col] for m in models]
        colors = ['#27AE60' if d >= 0 else '#E74C3C' for d in deltas]
        ax.bar(range(len(models)), deltas, color=colors, alpha=0.85)
        ax.axhline(0, color='black', lw=1)
        ax.set_xticks(range(len(models)))
        ax.set_xticklabels([m.replace('_se', '') for m in models],
                            rotation=30, ha='right', fontsize=8)
        ax.set_title(f"Δ {lab}\n(AUG − NO-AUG)", fontsize=10)
        ax.grid(axis='y', alpha=0.3)

    fig.suptitle('Augmentation Impact (Image-Level, positive = aug helps)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    _save(fig, os.path.join(save_dir, "aug_vs_noaug_delta.png"))


def plot_cv_metric_violin(summary_df, save_dir):
    rows = []
    for _, row in summary_df.iterrows():
        model = row['model']; cond = row['aug_condition']
        for metric in ['auc_pr', 'auc_roc', 'sensitivity', 'mcc']:
            key = f'{metric}_folds'
            if key in row and isinstance(row[key], list):
                for val in row[key]:
                    rows.append({'model': model, 'cond': cond,
                                  'metric': metric, 'value': val})
    if not rows: return

    df_v    = pd.DataFrame(rows)
    metrics = df_v['metric'].unique()
    fig, axes = plt.subplots(1, len(metrics), figsize=(5 * len(metrics), 6))
    if len(metrics) == 1: axes = [axes]

    for ax, met in zip(axes, metrics):
        sub = df_v[df_v['metric'] == met].copy()
        sub['label'] = sub['model'].str.replace('_se', '') + '\n' + sub['cond']
        order = sub['label'].unique()
        sns.violinplot(data=sub, x='label', y='value', ax=ax,
                       order=order, palette='Set2', inner='point')
        ax.set_title(met.upper().replace('_', ' '), fontsize=11)
        ax.set_xlabel(''); ax.set_ylabel('Score (Image-Level)')
        ax.tick_params(axis='x', labelsize=7)
        ax.grid(axis='y', alpha=0.3)

    fig.suptitle('Per-Fold Metric Distribution (Image-Level)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    _save(fig, os.path.join(save_dir, "cv_metric_violin.png"))


## 5-FOLD CV FOR ONE MODEL × AUG CONDITION


In [23]:

def run_cv(model_name, df, groups, aug_cond, device, root):
    save_dir = os.path.join(root, model_name, aug_cond)
    os.makedirs(save_dir, exist_ok=True)

    print(f"\n{'='*70}")
    print(f"  CV: {model_name.upper()} [{aug_cond}]")
    print(f"{'='*70}")

    y_all = np.array([cfg.CLASS_TO_LABEL.get(str(r['class']).lower(), 0)
                      for _, r in df.iterrows()])
    sgkf  = StratifiedGroupKFold(n_splits=cfg.N_FOLDS, shuffle=True,
                                  random_state=cfg.RANDOM_SEED)

    fold_mets, histories = [], []

    for fi, (tri, vai) in enumerate(sgkf.split(df, y_all, groups)):
        set_seed(cfg.RANDOM_SEED + fi)

        df_tr = df.iloc[tri].reset_index(drop=True)
        df_va = df.iloc[vai].reset_index(drop=True)

        overlap   = set(groups[tri]) & set(groups[vai])
        status    = "✅ no patient leakage" if not overlap else f"❌ {len(overlap)} overlap!"
        n_pat_tr  = len(set(groups[tri]))
        n_pat_va  = len(set(groups[vai]))
        print(f"  Fold {fi+1}: TR={len(df_tr)} ROIs ({n_pat_tr} patients) / "
              f"VA={len(df_va)} ROIs ({n_pat_va} patients) | {status}")

        tr_tfm = (get_train_transform_aug() if aug_cond == 'with_aug'
                  else get_noaug_transform())
        tr_ld  = make_loader(df_tr, tr_tfm, shuffle=True,
                             weighted=cfg.USE_WEIGHTED_SAMPLER)
        va_ld  = make_loader(df_va, get_val_transform(), shuffle=False)

        model = get_model(model_name).to(device)
        ckpt  = os.path.join(save_dir, f"fold{fi+1}_best.pth")

        hist = train_loop(model, tr_ld, va_ld, device,
                          cfg.CV_EPOCHS, ckpt, verbose=True,
                          aug_cond=aug_cond)
        histories.append(hist)

        if os.path.exists(ckpt):
            state = torch.load(ckpt, map_location=device, weights_only=False)
            model.load_state_dict(state['model_state_dict'])

        # ── Temperature scaling on validation set ─────────────────────────
        T = fit_temperature(model, va_ld, device)

        # ── Post-calibration sanity: verify T is reasonable ─────────────
        # T outside [0.1, 10.0] usually indicates a degenerate optimisation
        # (already protected by the log-parametrization; this is belt-and-
        # suspenders for any future changes).
        if not (0.1 <= T <= 10.0):
            print(f"     ⚠️  Fold {fi+1}: T={T:.4f} out of range — resetting to 1.0")
            T = 1.0

        # ── Collect calibrated val probs (image-level, max-pooled) ────────
        model.eval()
        raw_lbl, raw_prob, raw_ids = [], [], []
        with torch.no_grad():
            for imgs, lbls, ids in va_ld:
                logits = model(imgs.to(device))
                probs  = F.softmax(logits / T, dim=1)
                raw_prob.extend(probs.cpu().numpy())
                raw_lbl.extend(lbls.numpy())
                raw_ids.extend(ids)

        raw_lbl  = np.array(raw_lbl)
        raw_prob = np.array(raw_prob)

        # IMAGE-level aggregation (max pooling)
        img_lbl, img_prob = aggregate_rois(raw_lbl, raw_prob, raw_ids)

        # Optimal threshold on calibrated image-level val predictions
        opt_thr = find_optimal_threshold(img_lbl, img_prob[:, 1],
                                          criterion='youden')

        img_pred = (img_prob[:, 1] >= opt_thr).astype(int)
        fold_m   = compute_metrics(img_lbl, img_pred, img_prob)
        fold_m['optimal_threshold'] = opt_thr
        fold_m['temperature']       = T

        # Store for calibration plots
        fold_m['_y_true_raw']   = raw_lbl        # ROI-level labels
        fold_m['_y_prob_raw']   = raw_prob[:, 1] # ROI-level P(malignant)
        fold_m['_img_lbl_raw']  = img_lbl        # image-level labels
        fold_m['_img_prob_raw'] = img_prob[:, 1] # image-level P(malignant)

        fold_mets.append(fold_m)
        del model; torch.cuda.empty_cache()

        print(f"  → AUC-ROC={fold_m['auc_roc']:.4f}  AUC-PR={fold_m['auc_pr']:.4f}"
              f"  MCC={fold_m['mcc']:.4f}  Sens={fold_m['sensitivity']:.4f}"
              f"  Spec={fold_m['specificity']:.4f}  ECE={fold_m['ece']:.4f}"
              f"  Thr={opt_thr:.3f}  T={T:.3f}")

    agg = aggregate_cv_metrics(fold_mets)

    print(f"\n  {'─'*55}")
    print(f"  {model_name} [{aug_cond}] — 5-Fold Aggregate (Image-Level)")
    print(f"  {'─'*55}")
    for k in _SCALAR_KEYS:
        if f'{k}_mean' in agg:
            print(f"  {k:30s}: {agg[k+'_mean']:.4f} ± {agg[k+'_std']:.4f}")
    print(f"  {'─'*55}")

    prefix = f"{model_name} [{aug_cond}]"
    plot_roc_pr_curves(fold_mets, save_dir, prefix=prefix)
    plot_confusion_matrices(fold_mets, save_dir, prefix=prefix)
    plot_calibration(fold_mets, save_dir, prefix=prefix)
    plot_training_curves(histories[-1], save_dir,
                          prefix=f"{prefix} — Fold {cfg.N_FOLDS}")

    serial_folds = [{k: v for k, v in m.items() if not k.startswith('_')}
                    for m in fold_mets]
    with open(os.path.join(save_dir, "cv_metrics.json"), 'w') as f:
        json.dump({'aggregate': agg, 'per_fold': serial_folds},
                  f, indent=2, cls=NumpyEncoder)

    return {'aggregate': agg, 'per_fold': fold_mets}


# ============================================================================
# FINAL RETRAINING → HELD-OUT TEST
#
# FIX (CRITICAL): Internal val split is now patient-safe.
# We sample a fraction of PATIENTS (not ROI rows) to hold out, so that
# all ROIs of a given patient remain in the same partition.
# ============================================================================

def _patient_safe_val_split(df: pd.DataFrame,
                              val_patient_fraction: float = 0.10,
                              seed: int = 42) -> tuple:
    """
    Split df into (train_sub, val_sub) by sampling whole patients.

    Guarantees zero patient leakage between train_sub and val_sub:
      - Select val_patient_fraction of unique patient_ids as val patients.
      - ALL ROIs of those patients go into val_sub.
      - ALL other ROIs go into train_sub.

    Falls back to source_image-level split if patient_id is unavailable.
    """
    group_col = 'patient_id' if 'patient_id' in df.columns else 'source_image'

    unique_groups = df[group_col].unique()
    rng           = np.random.default_rng(seed)

    n_val_groups  = max(1, int(len(unique_groups) * val_patient_fraction))
    # Ensure minimum coverage
    n_val_groups  = max(n_val_groups, 4)

    val_groups = set(rng.choice(unique_groups, size=n_val_groups, replace=False))

    val_mask   = df[group_col].isin(val_groups)
    val_sub    = df[val_mask].reset_index(drop=True)
    train_sub  = df[~val_mask].reset_index(drop=True)

    # Verify no leakage
    overlap = set(train_sub[group_col]) & set(val_sub[group_col])
    assert len(overlap) == 0, (
        f"Patient leakage in _patient_safe_val_split: {overlap}")

    print(f"     Internal val split [{group_col}]: "
          f"{len(set(val_sub[group_col]))} val groups / "
          f"{len(set(train_sub[group_col]))} train groups | "
          f"val ROIs={len(val_sub)}  train ROIs={len(train_sub)} | "
          f"✅ no {group_col} leakage")
    return train_sub, val_sub


def final_retrain_and_test(best_name, best_cond,
                            train_val_df, test_df, device, root):
    print("\n" + "=" * 70)
    print(f"FINAL RETRAINING: {best_name} [{best_cond}]")
    print(f"  Full train+val : {len(train_val_df)} ROIs "
          f"({train_val_df['source_image'].nunique()} images)")
    print(f"  Test (held-out): {len(test_df)} ROIs "
          f"({test_df['source_image'].nunique()} images)")
    print("=" * 70)

    final_dir = os.path.join(root, "final_model")
    os.makedirs(final_dir, exist_ok=True)

    # ── FIX (CRITICAL): patient-safe internal val split ───────────────────
    tr_sub, val_sub = _patient_safe_val_split(
        train_val_df,
        val_patient_fraction=cfg.FINAL_VAL_PATIENT_FRACTION,
        seed=cfg.RANDOM_SEED)

    tr_tfm = (get_train_transform_aug() if best_cond == 'with_aug'
              else get_noaug_transform())
    tr_ld  = make_loader(tr_sub,  tr_tfm, shuffle=True,
                         weighted=cfg.USE_WEIGHTED_SAMPLER)
    va_ld  = make_loader(val_sub, get_val_transform(), shuffle=False)
    te_ld  = make_loader(test_df, get_val_transform(), shuffle=False)

    model = get_model(best_name).to(device)
    ckpt  = os.path.join(final_dir, "final_best.pth")

    hist = train_loop(model, tr_ld, va_ld, device,
                      cfg.FINAL_EPOCHS, ckpt, verbose=True,
                      aug_cond=best_cond)

    if os.path.exists(ckpt):
        state = torch.load(ckpt, map_location=device, weights_only=False)
        model.load_state_dict(state['model_state_dict'])
        print(f"  ✅ Best ckpt: epoch {state['epoch']}, "
              f"metric={state['best_metric']:.4f}")

    # ── Temperature scaling on internal (patient-safe) val set ───────────
    T = fit_temperature(model, va_ld, device)

    # ── Threshold optimisation on calibrated image-level val predictions ──
    model.eval()
    cal_lbl, cal_prob, cal_ids = [], [], []
    with torch.no_grad():
        for imgs, lbls, ids in va_ld:
            logits = model(imgs.to(device))
            probs  = F.softmax(logits / T, dim=1)
            cal_prob.extend(probs.cpu().numpy())
            cal_lbl.extend(lbls.numpy())
            cal_ids.extend(ids)

    cal_lbl  = np.array(cal_lbl)
    cal_prob = np.array(cal_prob)
    # IMAGE-level aggregation for threshold calibration
    val_img_lbl, val_img_prob = aggregate_rois(cal_lbl, cal_prob, cal_ids)
    opt_thr = find_optimal_threshold(val_img_lbl, val_img_prob[:, 1],
                                      criterion='youden')
    print(f"  ✅ Final model optimal threshold: {opt_thr:.4f}")
    print(f"  ✅ Final model temperature      : {T:.4f}")

    # ── Evaluate on held-out test set ─────────────────────────────────────
    model.eval()
    te_lbl, te_prob, te_ids = [], [], []
    with torch.no_grad():
        for imgs, lbls, ids in te_ld:
            logits = model(imgs.to(device))
            probs  = F.softmax(logits / T, dim=1)
            te_prob.extend(probs.cpu().numpy())
            te_lbl.extend(lbls.numpy())
            te_ids.extend(ids)

    te_lbl  = np.array(te_lbl)
    te_prob = np.array(te_prob)

    # IMAGE-level aggregation for final test evaluation
    img_lbl, img_prob = aggregate_rois(te_lbl, te_prob, te_ids)
    img_pred = (img_prob[:, 1] >= opt_thr).astype(int)

    test_m = compute_metrics(img_lbl, img_pred, img_prob)
    test_m['optimal_threshold'] = opt_thr
    test_m['temperature']       = T

    print("\n" + "=" * 70)
    print("FINAL TEST SET RESULTS — Image-Level (evaluated once on held-out data)")
    print(f"  Temperature: {T:.4f} | Threshold: {opt_thr:.4f}")
    print("=" * 70)
    for k in _SCALAR_KEYS:
        if k in test_m:
            print(f"  {k:30s}: {test_m[k]:.4f}")

    print("\nClassification Report (Image-Level):")
    cr = test_m['classification_report']
    for cls, vals in cr.items():
        if isinstance(vals, dict):
            print(f"  {cls:15s}  "
                  f"P={vals.get('precision',0):.3f}  "
                  f"R={vals.get('recall',0):.3f}  "
                  f"F1={vals.get('f1-score',0):.3f}  "
                  f"N={int(vals.get('support',0))}")

    prefix = f"Final Test — {best_name} [{best_cond}]"
    plot_training_curves(hist, final_dir, prefix=prefix)

    # Confusion matrix
    cm = np.array(test_m['confusion_matrix'])
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=cfg.CLASS_NAMES,
                yticklabels=cfg.CLASS_NAMES, ax=ax)
    tn, fp, fn, tp = cm.ravel()
    ax.set_title(f"Test CM (Image-Level) | "
                 f"Sens={tp/(tp+fn):.3f}  Spec={tn/(tn+fp):.3f}\n"
                 f"T={T:.3f}  thr={opt_thr:.3f}", fontsize=10)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    plt.tight_layout()
    _save(fig, os.path.join(final_dir, "test_confusion_matrix.png"))

    # ROC + PR
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].plot(test_m['fpr'], test_m['tpr'], 'b-', lw=2,
                 label=f"AUC-ROC={test_m['auc_roc']:.4f}")
    axes[0].plot([0, 1], [0, 1], '--', color='silver')
    axes[0].set(title='Test ROC (Image-Level)', xlabel='FPR', ylabel='TPR')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(test_m['recall_curve'], test_m['precision_curve'],
                 'g-', lw=2, label=f"AUC-PR={test_m['auc_pr']:.4f}")
    axes[1].set(title='Test PR Curve (Image-Level)', xlabel='Recall',
                ylabel='Precision')
    axes[1].legend(); axes[1].grid(alpha=0.3)
    fig.suptitle(prefix, fontsize=12, fontweight='bold')
    plt.tight_layout()
    _save(fig, os.path.join(final_dir, "test_roc_pr.png"))

    # Calibration (image-level, post-temperature-scaling)
    pt, pp = calibration_curve(img_lbl, img_prob[:, 1],
                                n_bins=cfg.CALIB_N_BINS,
                                strategy='uniform')
    ece_img = ece(img_lbl, img_prob[:, 1], cfg.CALIB_N_BINS)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot([0, 1], [0, 1], 'k--', label='Perfect')
    ax.plot(pp, pt, 'ro-', lw=2,
            label=f"Model (ECE={ece_img:.4f}, T={T:.3f})")
    ax.set(title=f"Test Calibration (Image-Level) — {best_name}",
           xlabel='Mean Predicted Prob', ylabel='Fraction of Positives')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    _save(fig, os.path.join(final_dir, "test_calibration.png"))

    serial = {k: v for k, v in test_m.items() if not k.startswith('_')}
    with open(os.path.join(final_dir, "test_metrics.json"), 'w') as f:
        json.dump(serial, f, indent=2, cls=NumpyEncoder)

    return test_m


## OVERFIT SANITY TEST


In [24]:

def overfit_sanity_test(df: pd.DataFrame, device: torch.device,
                        n_samples: int = 128, n_epochs: int = 30):
    print("\n" + "=" * 70)
    print(f"  OVERFIT SANITY TEST  ({n_samples} samples, {n_epochs} epochs)")
    print("=" * 70)

    ben = df[df['class'] == 'benign'].sample(
        min(n_samples // 2, (df['class'] == 'benign').sum()),
        random_state=cfg.RANDOM_SEED)
    mal = df[df['class'] == 'malignant'].sample(
        min(n_samples // 2, (df['class'] == 'malignant').sum()),
        random_state=cfg.RANDOM_SEED)
    sub = pd.concat([ben, mal]).sample(
        frac=1, random_state=cfg.RANDOM_SEED).reset_index(drop=True)

    print(f"  Subset: {len(sub)} rows — benign={len(ben)}  malignant={len(mal)}")

    loader = make_loader(sub, get_noaug_transform(),
                         shuffle=True, weighted=False)

    model = get_model('resnet18_se').to(device)
    for m in model.modules():
        if isinstance(m, nn.Dropout): m.p = 0.0
    unfreeze_backbone(model)

    opt = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    acc = 0.0
    for ep in range(n_epochs):
        model.train()
        correct = total = 0
        for imgs, lbls, _ in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            opt.zero_grad()
            out  = model(imgs)
            loss = criterion(out, lbls)
            loss.backward(); opt.step()
            correct += (out.argmax(1) == lbls).sum().item()
            total   += lbls.size(0)
        acc = correct / total
        if (ep + 1) % 10 == 0:
            print(f"  Ep {ep+1:3d} | train_acc={acc*100:.1f}%")

    if acc >= 0.80:
        print(f"  ✅ PASS — model can overfit ({acc*100:.1f}% ≥ 80%). "
              f"Data pipeline is healthy.")
    else:
        print(f"  ❌ FAIL — model cannot overfit ({acc*100:.1f}% < 80%). "
              f"Check labels, file paths, and preprocessing.")
    print("=" * 70)
    del model
    torch.cuda.empty_cache()


## MAIN


In [25]:

if __name__ == "__main__":

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n✅ Device : {device}")
    if torch.cuda.is_available():
        print(f"✅ GPU    : {torch.cuda.get_device_name(0)}")
    print(f"✅ PyTorch: {torch.__version__}\n")

    os.makedirs(cfg.RESULTS_DIR, exist_ok=True)

    # ── Load data ─────────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("LOADING DATASET")
    print("=" * 70)
    train_df, val_df, test_df = load_all_splits()

    # ── Audit each split ──────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("DATA AUDIT")
    print("=" * 70)
    train_df = audit_dataframe(train_df, 'train', check_files=True)
    val_df   = audit_dataframe(val_df,   'val',   check_files=True)
    test_df  = audit_dataframe(test_df,  'test',  check_files=True)

    print("\n  ROI-to-source-image grouping check:")
    for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
        _print_source_image_stats(df, name)

    for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
        n_b = (df['class'] == 'benign').sum()
        n_m = (df['class'] == 'malignant').sum()
        print(f"  {name:5s}: {len(df):5d} ROIs | "
              f"Benign={n_b} | Malignant={n_m} | "
              f"Images={df['source_image'].nunique()}")

    # ── Imbalance analysis ────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("CLASS IMBALANCE ANALYSIS")
    print("=" * 70)
    all_stats = {}
    for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
        all_stats[name] = compute_class_statistics(df, name)

    plot_imbalance_overview(all_stats, cfg.RESULTS_DIR)

    cv_df = pd.concat([train_df, val_df], ignore_index=True)
    compute_class_statistics(cv_df, 'cv_pool')

    # ── Load patient groups ───────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("LOADING PATIENT GROUPS")
    print("=" * 70)
    cv_groups = load_metadata_patient_groups(cv_df)

    # ── Overfit sanity test ───────────────────────────────────────────────
    overfit_sanity_test(cv_df, device, n_samples=128, n_epochs=30)

    # ── CV grid ───────────────────────────────────────────────────────────
    aug_conditions  = ['with_aug', 'no_aug']
    cv_summary_rows = []

    for model_name, aug_cond in it_product(cfg.MODEL_NAMES, aug_conditions):
        try:
            res = run_cv(model_name, cv_df, cv_groups, aug_cond,
                         device, cfg.RESULTS_DIR)
            row = {'model': model_name, 'aug_condition': aug_cond}
            row.update(res['aggregate'])
            cv_summary_rows.append(row)
        except Exception as e:
            print(f"\n❌ Failed {model_name} [{aug_cond}]: {e}")
            import traceback; traceback.print_exc()

    # ── Summary table ─────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("CV SUMMARY (Image-Level Metrics)")
    print("=" * 70)

    summary_df   = pd.DataFrame(cv_summary_rows)
    display_cols = (
        ['model', 'aug_condition'] +
        [f'{m}_{s}' for m in ['auc_roc', 'auc_pr', 'f1_macro',
                               'sensitivity', 'specificity', 'mcc',
                               'brier_score', 'ece', 'optimal_threshold']
         for s in ['mean', 'std']]
    )
    avail = [c for c in display_cols if c in summary_df.columns]
    print(summary_df[avail].to_string(index=False))

    csv_path = os.path.join(cfg.RESULTS_DIR, "cv_summary.csv")
    summary_df.to_csv(csv_path, index=False)
    print(f"\n✅ CV summary: {csv_path}")

    # ── Plots ─────────────────────────────────────────────────────────────
    if len(summary_df) > 0:
        plot_cross_model_comparison(summary_df, cfg.RESULTS_DIR)
        plot_aug_vs_noaug(summary_df, cfg.RESULTS_DIR)
        plot_cv_metric_violin(summary_df, cfg.RESULTS_DIR)

    # ── Select best model ─────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("MODEL SELECTION")
    print("=" * 70)

    if len(summary_df) > 0 and 'auc_pr_mean' in summary_df.columns:
        best_row  = summary_df.loc[summary_df['auc_pr_mean'].idxmax()]
        best_name = best_row['model']
        best_cond = best_row['aug_condition']
        print(f"  ✅ Best model    : {best_name}")
        print(f"  ✅ Aug condition : {best_cond}")
        print(f"  ✅ CV AUC-PR     : {best_row['auc_pr_mean']:.4f} "
              f"± {best_row['auc_pr_std']:.4f}")
    else:
        best_name, best_cond = cfg.MODEL_NAMES[0], 'with_aug'
        print(f"  ⚠️  Defaulting to {best_name} [{best_cond}]")

    # ── Final retrain → test ───────────────────────────────────────────────
    test_res = final_retrain_and_test(
        best_name, best_cond, cv_df, test_df, device, cfg.RESULTS_DIR)

    # ── Final summary ──────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("🎉  PIPELINE COMPLETE")
    print("=" * 70)
    print(f"\n  Best CV model : {best_name} [{best_cond}]")
    print(f"\n  HELD-OUT TEST RESULTS (Image-Level):")
    for label, key in [
        ('AUC-ROC',      'auc_roc'),
        ('AUC-PR',       'auc_pr'),
        ('Accuracy',     'accuracy'),
        ('Balanced Acc', 'balanced_accuracy'),
        ('Sensitivity',  'sensitivity'),
        ('Specificity',  'specificity'),
        ('F1-Macro',     'f1_macro'),
        ('MCC',          'mcc'),
        ('Brier Score',  'brier_score'),
        ('ECE',          'ece'),
        ('Threshold',    'optimal_threshold'),
        ('Temperature',  'temperature'),
    ]:
        if key in test_res:
            print(f"  {label:20s}: {test_res[key]:.4f}")
    print(f"\n  📁 Results: {cfg.RESULTS_DIR}/")
    print("=" * 70)



✅ Device : cuda
✅ GPU    : Tesla T4
✅ PyTorch: 2.9.0+cu126


LOADING DATASET
  📂 Loading metadata: /kaggle/input/datasets/blaster21/5foldmetadata/roi_metadata_with_patient.csv

  source_image check (should match roi_filepath basename exactly):
  roi_filepath basename                          -> source_image
  --------------------------------------------      ----------------------------------
  IMG000001_roi1_other-mt_ok_iou0.47_conf0.672.jpg  -> IMG000001_roi1_other-mt_ok_iou0.47_conf0.672.jpg  OK
  IMG000006_roi0_osteosarcoma_ok_iou0.76_conf0.923.jpg  -> IMG000006_roi0_osteosarcoma_ok_iou0.76_conf0.923.jpg  OK
  IMG000007_roi0_osteosarcoma_ok_iou0.78_conf0.937.jpg  -> IMG000007_roi0_osteosarcoma_ok_iou0.78_conf0.937.jpg  OK
  IMG000008_roi0_osteosarcoma_ok_iou0.78_conf0.894.jpg  -> IMG000008_roi0_osteosarcoma_ok_iou0.78_conf0.894.jpg  OK
  IMG000009_roi0_osteosarcoma_ok_iou0.79_conf0.910.jpg  -> IMG000009_roi0_osteosarcoma_ok_iou0.79_conf0.910.jpg  OK


DATA AUDIT

  DATA AUDIT [TRA

100%|██████████| 44.7M/44.7M [00:00<00:00, 189MB/s]


  ✅ resnet18_se: 11.4M total / 11.4M trainable
  Ep  10 | train_acc=89.8%
  Ep  20 | train_acc=100.0%
  Ep  30 | train_acc=100.0%
  ✅ PASS — model can overfit (100.0% ≥ 80%). Data pipeline is healthy.

  CV: DENSENET121_SE [with_aug]
  Fold 1: TR=1389 ROIs (501 patients) / VA=426 ROIs (129 patients) | ✅ no patient leakage
     Sampler: benign≈50.0%  malignant≈50.0%  (target 50/50)
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 190MB/s]


  ✅ densenet121_se: 7.3M total / 7.3M trainable
     Ep  5 | TL=0.4779 TA=85.3% | VL=0.3523 AUC-PR=0.6591 | Pat=0/10
     Ep 10 | TL=0.4695 TA=88.2% | VL=0.4224 AUC-PR=0.6820 | Pat=0/10
     Ep 15 | TL=0.4685 TA=87.5% | VL=0.3467 AUC-PR=0.6616 | Pat=0/10
     🔓 Unfreeze @ ep 16 (BN layers kept frozen)
     Ep 20 | TL=0.4162 TA=91.8% | VL=0.3300 AUC-PR=0.7220 | Pat=0/10
     Ep 25 | TL=0.3977 TA=91.9% | VL=0.3143 AUC-PR=0.7526 | Pat=0/10
     Ep 30 | TL=0.3899 TA=92.2% | VL=0.3060 AUC-PR=0.7640 | Pat=0/10
     Ep 35 | TL=0.4328 TA=91.6% | VL=0.3105 AUC-PR=0.7494 | Pat=4/10
     Ep 40 | TL=0.3966 TA=94.0% | VL=0.3099 AUC-PR=0.7533 | Pat=9/10
     Temperature scaling: T=0.6408
     🎯 Optimal threshold (Youden): 0.0527  (Sens=0.974, Spec=0.793)
  → AUC-ROC=0.9587  AUC-PR=0.7677  MCC=0.4947  Sens=0.9744  Spec=0.7933  ECE=0.0383  Thr=0.053  T=0.641
  Fold 2: TR=1482 ROIs (497 patients) / VA=333 ROIs (133 patients) | ✅ no patient leakage
     Sampler: benign≈50.0%  malignant≈50.0%  (target 50

...

     Ep  5 | TL=0.6500 TA=64.9% | VL=0.5203 AUC-PR=0.5348 | Pat=0/10
     Ep 10 | TL=0.6018 TA=72.6% | VL=0.5217 AUC-PR=0.5367 | Pat=0/10
     Ep 15 | TL=0.5945 TA=72.6% | VL=0.5625 AUC-PR=0.5459 | Pat=0/10
     🔓 Unfreeze @ ep 16 (BN layers kept frozen)
     Ep 20 | TL=0.5619 TA=77.0% | VL=0.4903 AUC-PR=0.5620 | Pat=0/10
     Ep 25 | TL=0.5139 TA=82.1% | VL=0.4620 AUC-PR=0.5959 | Pat=1/10
     Ep 30 | TL=0.5122 TA=81.5% | VL=0.4402 AUC-PR=0.6182 | Pat=0/10
     Ep 35 | TL=0.5212 TA=82.7% | VL=0.4185 AUC-PR=0.6423 | Pat=0/10
     Ep 40 | TL=0.4991 TA=84.4% | VL=0.4603 AUC-PR=0.6677 | Pat=0/10
     Temperature scaling: T=0.8108
     🎯 Optimal threshold (Youden): 0.7527  (Sens=0.641, Spec=0.948)
  → AUC-ROC=0.8731  AUC-PR=0.6677  MCC=0.5530  Sens=0.6410  Spec=0.9483  ECE=0.2091  Thr=0.753  T=0.811
  Fold 2: TR=1482 ROIs (497 patients) / VA=333 ROIs (133 patients) | ✅ no patient leakage
     Sampler: benign≈50.0%  malignant≈50.0%  (target 50/50)
  ✅ mobilenet_v2_se: 2.8M total / 2.8M trai

100%|██████████| 109M/109M [00:00<00:00, 220MB/s] 


  ✅ convnext_tiny_se: 28.1M total / 28.1M trainable
     Ep  5 | TL=0.5963 TA=72.3% | VL=0.3953 AUC-PR=0.6650 | Pat=0/10
     Ep 10 | TL=0.5604 TA=76.7% | VL=0.4398 AUC-PR=0.6704 | Pat=0/10
     Ep 15 | TL=0.5544 TA=77.0% | VL=0.3838 AUC-PR=0.6889 | Pat=0/10
     🔓 Unfreeze @ ep 16 (BN layers kept frozen)
     Ep 20 | TL=0.5083 TA=82.4% | VL=0.4035 AUC-PR=0.7177 | Pat=0/10
     Ep 25 | TL=0.4697 TA=85.1% | VL=0.3761 AUC-PR=0.7601 | Pat=1/10
     Ep 30 | TL=0.4644 TA=86.0% | VL=0.3636 AUC-PR=0.7631 | Pat=6/10
     Ep 35 | TL=0.4612 TA=88.3% | VL=0.3744 AUC-PR=0.7765 | Pat=2/10
     Ep 40 | TL=0.4378 TA=90.4% | VL=0.3560 AUC-PR=0.7837 | Pat=1/10
     Temperature scaling: T=0.7022
     🎯 Optimal threshold (Youden): 0.1916  (Sens=0.974, Spec=0.811)
  → AUC-ROC=0.9606  AUC-PR=0.7948  MCC=0.5162  Sens=0.9744  Spec=0.8114  ECE=0.0951  Thr=0.192  T=0.702
  Fold 2: TR=1482 ROIs (497 patients) / VA=333 ROIs (133 patients) | ✅ no patient leakage
     Sampler: benign≈50.0%  malignant≈50.0%  (targe

100%|██████████| 35.2M/35.2M [00:00<00:00, 187MB/s]


  ✅ efficientnet_b2_se: 8.3M total / 8.3M trainable
     Ep  5 | TL=0.5131 TA=81.2% | VL=0.4575 AUC-PR=0.6317 | Pat=0/10
     Ep 10 | TL=0.4626 TA=87.8% | VL=0.4120 AUC-PR=0.6840 | Pat=0/10
     Ep 15 | TL=0.4402 TA=89.8% | VL=0.3766 AUC-PR=0.6937 | Pat=0/10
     🔓 Unfreeze @ ep 16 (BN layers kept frozen)
     Ep 20 | TL=0.4478 TA=89.0% | VL=0.3863 AUC-PR=0.7055 | Pat=0/10
     Ep 25 | TL=0.4278 TA=88.9% | VL=0.3631 AUC-PR=0.6826 | Pat=5/10
     Ep 30 | TL=0.4283 TA=89.6% | VL=0.3801 AUC-PR=0.6939 | Pat=10/10
     ⛔ Early stop ep30 | best=0.7055
     Temperature scaling: T=0.7158
     🎯 Optimal threshold (Youden): 0.5779  (Sens=0.795, Spec=0.917)
  → AUC-ROC=0.9294  AUC-PR=0.7055  MCC=0.5786  Sens=0.7949  Spec=0.9173  ECE=0.1318  Thr=0.578  T=0.716
  Fold 2: TR=1482 ROIs (497 patients) / VA=333 ROIs (133 patients) | ✅ no patient leakage
     Sampler: benign≈50.0%  malignant≈50.0%  (target 50/50)
  ✅ efficientnet_b2_se: 8.3M total / 8.3M trainable
     Ep  5 | TL=0.5186 TA=80.3% | VL=0.

In [26]:
import shutil

source_folder = "/kaggle/working/"
zip_path = "/kaggle/working/class_final_chrome_better.zip"

shutil.make_archive(
    base_name=zip_path.replace(".zip",""),
    format="zip",
    root_dir=source_folder
)

print("✅ Zipped successfully:", zip_path)


✅ Zipped successfully: /kaggle/working/class_final_chrome_better.zip


In [27]:
# ============================================================
# CELL 1 — Run this FIRST, before any other imports
# Kaggle-safe install: Captum + pytorch-grad-cam
# No version conflicts with Kaggle's pre-installed PyTorch
# ============================================================

import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-800:])
    else:
        print("OK:", cmd)
    return result.returncode

# Step 1: upgrade pip silently
run(f"{sys.executable} -m pip install --upgrade pip -q")

# Step 2: install captum WITHOUT touching torch/torchvision
# --no-deps skips ALL dependency resolution → no conflict
run(f"{sys.executable} -m pip install captum --no-deps -q")

# Step 3: captum needs only these two extras (already on Kaggle)
# numpy and torch are already present, so just ensure these:
run(f"{sys.executable} -m pip install matplotlib -q")

# Step 4: pytorch-grad-cam (safe, no torch dep conflict)
run(f"{sys.executable} -m pip install grad-cam -q")

# ============================================================
# CELL 2 — Verify installs (run after Cell 1)
# ============================================================

import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

try:
    from captum.attr import IntegratedGradients
    print("✅ Captum OK")
except Exception as e:
    print("❌ Captum failed:", e)

try:
    from pytorch_grad_cam import GradCAMPlusPlus
    print("✅ pytorch-grad-cam OK")
except Exception as e:
    print("❌ pytorch-grad-cam failed:", e)




OK: /usr/bin/python3 -m pip install --upgrade pip -q
OK: /usr/bin/python3 -m pip install captum --no-deps -q
OK: /usr/bin/python3 -m pip install matplotlib -q
OK: /usr/bin/python3 -m pip install grad-cam -q
PyTorch: 2.9.0+cu126
CUDA: True
✅ Captum OK
✅ pytorch-grad-cam OK


In [28]:
"""
================================================================================
STAGE 3C — EXPLAINABILITY PIPELINE  v4  (Q1 camera-ready, Kaggle-ready)
================================================================================
Model    : TumorClassifierConvNeXtTinySE  (Benign=0 / Malignant=1)
Checkpoint: /kaggle/input/models/samiulalim172/cnn/pytorch/default/1/final_best.pth
Dataset  : BTXRD ROI Dataset  (test split only, ≥20 samples)

Methods
───────
  1. Integrated Gradients (IG)  — Captum       [pixel-level, axiomatic, signed]
  2. Grad-CAM++                 — pytorch-grad-cam  [spatial, gradient-weighted]

Five Publication Figures
────────────────────────
  fig1_malignant_correct.png    Best malignant (correct, highest P(Mal))
  fig2_benign_correct.png       Best benign    (correct, lowest  P(Mal))
  fig3_confident_misclass.png   Most confident misclassification
  fig4_borderline_case.png      Most uncertain prediction |P(Mal)-0.5| minimal
  fig5_random_representative.png  Seeded random balanced sample

Layout per figure (2 rows × 3 cols, equal cell sizes, clean margins)
──────────────────────────────────────────────────────────────────────
  Row 0 │ Original ROI  │ IG signed map (RdBu_r)       │ Grad-CAM++ heatmap (inferno)
  Row 1 │ Original ROI  │ IG overlay    (inferno, α)   │ Grad-CAM++ overlay + gold contour
  One colorbar per attribution column, top row only — row 1 shares scale.

Active Fixes
─────────────
  FIX-1  Dual-target: explain class-1 (malignant) AND predicted class
  FIX-2  Grad-CAM++ targets model.features[-1][-1] (last ConvNeXt block)
  FIX-3  IG baseline = Gaussian-blurred input (σ=10 px, in-distribution)
  FIX-5  Balanced sample selection (50/50); explicit illustrative disclaimers
  FIX-6  N_IG_STEPS=200; IG scalar smoothed with σ=1.0 px before display
  FIX-7  Structured geometry (inches→fractions); no colorbar overlap
  FIX-8  [NEW] Compact figures: per-row metadata strips prevent text overlap

Kaggle install (run once before this script):
  import subprocess, sys
  subprocess.run(f"{sys.executable} -m pip install captum --no-deps -q", shell=True)
  subprocess.run(f"{sys.executable} -m pip install grad-cam -q", shell=True)
================================================================================
"""

# ============================================================================
# 1.  IMPORTS
# ============================================================================

import os, re, json, random, warnings
from pathlib import Path
from collections import Counter
from typing import List, Tuple, Optional, Dict

import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
from matplotlib.colors import TwoSlopeNorm, Normalize
import matplotlib.patches as mpatches
import seaborn as sns

from PIL import Image, ImageOps

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T

warnings.filterwarnings('ignore')

# ── Captum ───────────────────────────────────────────────────────────────────
try:
    from captum.attr import IntegratedGradients
    CAPTUM_OK = True
except ImportError:
    CAPTUM_OK = False
    print("⚠️  Captum not found — run: pip install captum --no-deps -q")

# ── pytorch-grad-cam ─────────────────────────────────────────────────────────
try:
    from pytorch_grad_cam import GradCAMPlusPlus
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    GRADCAM_OK = True
except ImportError:
    GRADCAM_OK = False
    print("⚠️  pytorch-grad-cam not found — run: pip install grad-cam -q")


# ============================================================================
# 2.  CONFIGURATION
# ============================================================================

class Config:
    # ── Paths ────────────────────────────────────────────────────────────────
    CHECKPOINT_PATH = (
        "/kaggle/working/results_stage3c_cv/final_model/final_best.pth"
    )
    ROI_DATASET_DIR = (
        "/kaggle/input/datasets/sadibhasan/roi-dataset"
    )
    OUTPUT_DIR = "/kaggle/working/explainability_final"

    # ── Model ────────────────────────────────────────────────────────────────
    IMAGE_SIZE    = 256
    NUM_CLASSES   = 2
    SE_REDUCTION  = 16
    MALIGNANT_IDX = 1

    # ── IG ───────────────────────────────────────────────────────────────────
    N_IG_STEPS      = 100          # FIX-6: increased from 50 → 200
    IG_BASELINE     = 'blur'       # 'blur' | 'mean' | 'zero'
    BLUR_SIGMA      = 10.0         # Gaussian σ for blurred IG baseline (px)
    IG_SMOOTH_SIGMA = 1.0          # FIX-6: post-smoothing on IG scalar map

    # ── Sampling ─────────────────────────────────────────────────────────────
    N_SAMPLES     = 20             # total test ROIs to explain (≥20)

    # ── Visualisation ────────────────────────────────────────────────────────
    DPI           = 300
    FMT           = 'png'
    ALPHA         = 0.40           # reduced to 0.40 for natural X-ray visibility
    CLASS_NAMES   = ['Benign', 'Malignant']
    C2L           = {'benign': 0, 'malignant': 1}
    L2C           = {0: 'benign', 1: 'malignant'}
    RANDOM_SEED   = 42


cfg = Config()

# ── Colormaps ─────────────────────────────────────────────────────────────────
_IG_CMAP  = plt.cm.RdBu_r    # signed diverging: red=promotes, blue=inhibits
_CAM_CMAP = plt.cm.inferno    # sequential: activation intensity

# ── Typography (Q1 journal: 8–10 pt) — cleaner hierarchy
_FONT_TITLE  = 10   # main suptitle
_FONT_COLHDR = 8    # column headers
_FONT_BODY   = 7.5  # body / metadata
_FONT_ANNOT  = 6.5  # in-panel annotations
_FONT_CB     = 6    # colorbar labels
_FONT_SUBLBL = 6    # sub-labels under columns


# ============================================================================
# 3.  ARCHITECTURE  (must match checkpoint exactly)
# ============================================================================

class SEBlock(nn.Module):
    """Squeeze-and-Excitation recalibration block."""
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Sequential(
            nn.Linear(channels, mid, bias=False), nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False), nn.Sigmoid())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c = x.shape[:2]
        return x * self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1)


class TumorClassifierConvNeXtTinySE(nn.Module):
    """ConvNeXt-Tiny + global SE block. Output before GAP: (B, 768, H', W')."""
    def __init__(self, n: int = 2, r: int = 16):
        super().__init__()
        bb            = torchvision.models.convnext_tiny(weights=None)
        self.features = bb.features
        self.se       = SEBlock(768, r)
        self.pool     = nn.AdaptiveAvgPool2d(1)
        self.head     = nn.Sequential(
            nn.Flatten(), nn.LayerNorm(768),
            nn.Dropout(0.5), nn.Linear(768, 256), nn.GELU(),
            nn.Dropout(0.4), nn.Linear(256, n))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool(self.se(self.features(x)))
        return self.head(x)


# ============================================================================
# 4.  CHECKPOINT LOADER
# ============================================================================

def load_model(path: str, device: torch.device) -> nn.Module:
    print(f"\n{'='*60}\n  LOADING CHECKPOINT\n{'='*60}")
    print(f"  {path}")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Checkpoint not found: {path}")

    model = TumorClassifierConvNeXtTinySE(n=cfg.NUM_CLASSES, r=cfg.SE_REDUCTION)
    ckpt  = torch.load(path, map_location=device, weights_only=False)
    state = ckpt.get('model_state_dict', ckpt)
    miss, unexp = model.load_state_dict(state, strict=True)

    model.to(device).eval()
    assert not model.training, "model.training must be False after .eval()"

    epoch  = ckpt.get('epoch', '?')
    metric = ckpt.get('best_metric', float('nan'))
    npar   = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  ✅ epoch={epoch}  best_metric={metric:.4f}  params={npar:.1f}M")
    print(f"  ✅ missing={len(miss)}  unexpected={len(unexp)}")

    try:
        blk = model.features[-1][-1]
        print(f"  ✅ Grad-CAM++ layer: model.features[-1][-1] → {type(blk).__name__}")
    except Exception:
        print("  ⚠️  model.features[-1] not subscriptable; will use [-1]")
    print(f"{'='*60}\n")
    return model


# ============================================================================
# 5.  PRE-PROCESSING
# ============================================================================

_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]
_TFM  = T.Compose([T.ToTensor(), T.Normalize(_MEAN, _STD)])


def _autocontrast(img: Image.Image) -> Image.Image:
    img = img.convert('L')
    img = ImageOps.autocontrast(img, cutoff=1)
    return img.convert('RGB')


def _pad_resize(img: Image.Image, sz: int = 256) -> Image.Image:
    w, h   = img.size
    r      = min(sz / w, sz / h)
    nw, nh = int(w * r), int(h * r)
    img    = img.resize((nw, nh), Image.BILINEAR)
    out    = Image.new('RGB', (sz, sz), 0)
    out.paste(img, ((sz - nw) // 2, (sz - nh) // 2))
    return out


def preprocess(path: str) -> Tuple[torch.Tensor, np.ndarray]:
    img    = Image.open(path).convert('RGB')
    img    = _autocontrast(img)
    img    = _pad_resize(img, cfg.IMAGE_SIZE)
    rgb_u8 = np.array(img, dtype=np.uint8)
    tensor = _TFM(img).unsqueeze(0)
    return tensor, rgb_u8


def to_float_rgb(rgb_u8: np.ndarray) -> np.ndarray:
    return rgb_u8.astype(np.float32) / 255.


# ============================================================================
# 6.  IG BASELINE
# ============================================================================

def make_baseline(tensor  : torch.Tensor,
                  strategy: str   = 'blur',
                  sigma   : float = 10.0) -> torch.Tensor:
    if strategy in ('zero', 'mean'):
        return torch.zeros_like(tensor).requires_grad_(False)

    if strategy == 'blur':
        np_img  = tensor.squeeze(0).detach().cpu().numpy()
        blurred = np.stack(
            [gaussian_filter(np_img[c], sigma=sigma) for c in range(3)],
            axis=0)
        return (torch.from_numpy(blurred.copy()).unsqueeze(0)
                .to(dtype=tensor.dtype, device=tensor.device)
                .requires_grad_(False))

    raise ValueError(f"Unknown baseline: '{strategy}'.")


# ============================================================================
# 7.  METHOD 1 — INTEGRATED GRADIENTS
# ============================================================================

def compute_ig(model   : nn.Module,
               tensor  : torch.Tensor,
               target  : int,
               device  : torch.device,
               n_steps : int  = 200,
               baseline: str  = 'blur') -> Tuple[np.ndarray, float]:
    if not CAPTUM_OK:
        raise RuntimeError("pip install captum --no-deps -q")

    inp  = tensor.to(device).requires_grad_(True)
    base = make_baseline(inp, strategy=baseline, sigma=cfg.BLUR_SIGMA)

    ig          = IntegratedGradients(model)
    attr, delta = ig.attribute(
        inp,
        baselines                = base,
        target                   = target,
        n_steps                  = n_steps,
        method                   = 'gausslegendre',
        return_convergence_delta = True,
    )
    attr_signed = attr.squeeze(0).detach().cpu().numpy().astype(np.float32)
    delta_val   = float(delta.abs().mean().item())
    return attr_signed, delta_val


# ============================================================================
# 8.  METHOD 2 — GRAD-CAM++
# ============================================================================

def _gradcam_layer(model: nn.Module) -> list:
    try:
        return [model.features[-1][-1]]
    except (IndexError, TypeError):
        return [model.features[-1]]


def compute_gradcam_pp(model  : nn.Module,
                       tensor : torch.Tensor,
                       target : int,
                       device : torch.device) -> np.ndarray:
    if not GRADCAM_OK:
        raise RuntimeError("pip install grad-cam -q")

    model.eval()
    cam_obj = GradCAMPlusPlus(model=model, target_layers=_gradcam_layer(model))
    cam     = cam_obj(
        input_tensor = tensor.to(device),
        targets      = [ClassifierOutputTarget(target)],
    )
    raw = cam[0].astype(np.float32)
    return np.power(raw, 0.7)


# ============================================================================
# 9.  ACTIVATION CONCENTRATION
# ============================================================================

def activation_concentration(cam     : np.ndarray,
                              top_pct : float = 25.0) -> Tuple[float, np.ndarray]:
    threshold = np.percentile(cam, 100 - top_pct)
    mask      = cam >= threshold
    total     = cam.sum()
    conc      = float(cam[mask].sum() / total * 100) if total > 1e-8 else 0.0
    return conc, mask


# ============================================================================
# 10.  NORMALISATION HELPERS
# ============================================================================

def _norm_signed(attr        : np.ndarray,
                 smooth_sigma: float = 0.0) -> np.ndarray:
    scalar = attr.sum(axis=0)
    if smooth_sigma > 0:
        scalar = gaussian_filter(scalar, sigma=smooth_sigma)
    vmax = np.percentile(np.abs(scalar), 99)
    if vmax < 1e-8:
        return np.zeros_like(scalar, dtype=np.float32)
    return np.clip(scalar / vmax, -1, 1).astype(np.float32)


def _norm_abs(attr        : np.ndarray,
              smooth_sigma: float = 0.0) -> np.ndarray:
    scalar = np.abs(attr).sum(axis=0)
    if smooth_sigma > 0:
        scalar = gaussian_filter(scalar, sigma=smooth_sigma)
    vmax = np.percentile(scalar, 99)
    if vmax < 1e-8:
        return np.zeros_like(scalar, dtype=np.float32)
    return np.clip(scalar / vmax, 0, 1).astype(np.float32)


def _blend(rgb      : np.ndarray,
           attr_01  : np.ndarray,
           cmap,
           alpha    : float = 0.40) -> np.ndarray:
    hm = cmap(attr_01)[:, :, :3].astype(np.float32)
    return np.clip((1 - alpha) * rgb + alpha * hm, 0, 1)


# ============================================================================
# 11.  LABEL / VERDICT HELPERS
# ============================================================================

def _lstr(idx: int) -> str:
    return cfg.CLASS_NAMES[idx] if idx in (0, 1) else f'cls-{idx}'


def _verdict(r: dict) -> Tuple[str, str]:
    ok = r['label'] == r['pred_class']
    return ('✓', '#1a7a3c') if ok else ('✗', '#c0392b')


def _fmt_prob(p: float) -> str:
    if p < 0.001:
        return "P(Mal)<0.001"
    if p > 0.999:
        return "P(Mal)>0.999"
    return f"P(Mal)={p:.3f}"


def _meta_line(r: dict) -> str:
    sym, _ = _verdict(r)
    return (f"GT: {_lstr(r['label'])}  |  Pred: {_lstr(r['pred_class'])}  "
            f"|  {_fmt_prob(r['pred_prob'])}  |  {sym}")


# ============================================================================
# 12.  PER-SAMPLE INFERENCE + ATTRIBUTION
# ============================================================================

def explain_one(model  : nn.Module,
                path   : str,
                label  : int,
                device : torch.device) -> Optional[Dict]:
    try:
        tensor, rgb_u8 = preprocess(path)
    except Exception as e:
        print(f"  ⚠️  {Path(path).name}: {e}")
        return None

    model.eval()
    with torch.no_grad():
        logits = model(tensor.to(device))
        probs  = F.softmax(logits, dim=1).cpu().numpy()[0]

    pred     = int(np.argmax(probs))
    prob_mal = float(probs[cfg.MALIGNANT_IDX])

    result = dict(
        path       = path,
        label      = label,
        pred_class = pred,
        pred_prob  = prob_mal,
        confidence = float(probs[pred]),
        rgb_u8     = rgb_u8,
        tensor     = tensor,
    )

    targets = {'mal': cfg.MALIGNANT_IDX}
    if pred != cfg.MALIGNANT_IDX:
        targets['pred'] = pred

    for tname, tidx in targets.items():
        print(f"     IG + Grad-CAM++  target='{tname}' (class {tidx})")

        if CAPTUM_OK:
            try:
                ig_signed, delta = compute_ig(
                    model, tensor, tidx, device,
                    n_steps  = cfg.N_IG_STEPS,
                    baseline = cfg.IG_BASELINE)
                result[f'ig_signed_{tname}'] = ig_signed
                result[f'ig_delta_{tname}']  = delta
                print(f"       IG  Δ={delta:.5f}")
            except Exception as e:
                print(f"       IG failed: {e}")

        if GRADCAM_OK:
            try:
                cam         = compute_gradcam_pp(model, tensor, tidx, device)
                conc, mask  = activation_concentration(cam, top_pct=25)
                result[f'gpp_cam_{tname}']  = cam
                result[f'gpp_conc_{tname}'] = conc
                result[f'gpp_mask_{tname}'] = mask
                print(f"       Grad-CAM++  conc={conc:.1f}%")
            except Exception as e:
                print(f"       Grad-CAM++ failed: {e}")

    if pred == cfg.MALIGNANT_IDX:
        for key in list(result):
            if key.endswith('_mal'):
                result[key.replace('_mal', '_pred')] = result[key]

    return result


# ============================================================================
# 13.  EXEMPLAR SELECTOR
# ============================================================================

def _select_exemplars(results: list) -> dict:
    correct = [r for r in results if r['label'] == r['pred_class']]
    wrong   = [r for r in results if r['label'] != r['pred_class']]

    mal_c  = sorted([r for r in correct if r['label'] == 1],
                     key=lambda r: r['pred_prob'], reverse=True)
    ben_c  = sorted([r for r in correct if r['label'] == 0],
                     key=lambda r: r['pred_prob'])
    conf_w = sorted(wrong, key=lambda r: r['confidence'], reverse=True)
    border = sorted(results, key=lambda r: abs(r['pred_prob'] - 0.5))

    rng    = np.random.default_rng(cfg.RANDOM_SEED + 99)
    pool   = results[:]
    rng.shuffle(pool)

    return {
        'mal_correct'    : mal_c[0]   if mal_c   else None,
        'ben_correct'    : ben_c[0]   if ben_c   else None,
        'confident_wrong': conf_w[0]  if conf_w  else None,
        'borderline'     : border[0]  if border  else None,
        'random_rep'     : pool[0]    if pool     else None,
    }


# ============================================================================
# 14.  CORE FIGURE RENDERER  (2-row × 3-col per-sample figures)
# ============================================================================

def _render_figure(result     : dict,
                   figure_tag : str,
                   save_path  : str,
                   smooth_sig : float = 1.0,
                   alpha      : float = 0.40) -> None:
    rgb    = to_float_rgb(result['rgb_u8'])

    ig_raw = result.get('ig_signed_mal')
    cam    = result.get('gpp_cam_mal')
    conc   = result.get('gpp_conc_mal')
    mask   = result.get('gpp_mask_mal')

    ig_sc  = (_norm_signed(ig_raw, smooth_sigma=smooth_sig)
              if ig_raw is not None else None)
    ig_abs = (_norm_abs(ig_raw,    smooth_sigma=smooth_sig)
              if ig_raw is not None else None)

    sym, bc = _verdict(result)

    cell    = 2.80
    cb_h    = 0.16
    hdr_h   = 0.38
    sublbl_h= 0.18
    meta_h  = 0.26
    lr_mar  = 0.015

    fig_w   = cell * 3
    fig_h   = cell * 2 + cb_h + hdr_h + sublbl_h + meta_h

    f_meta   = meta_h   / fig_h
    f_hdr    = hdr_h    / fig_h
    f_cb     = cb_h     / fig_h
    f_row    = cell     / fig_h
    f_sublbl = sublbl_h / fig_h

    top_r0  = 1.0 - f_meta - f_hdr
    bot_r0  = top_r0 - f_row
    bot_cb  = bot_r0 - f_cb
    bot_r1  = bot_cb - f_row

    col_w   = (1.0 - 2 * lr_mar) / 3
    col_l   = [lr_mar + i * col_w for i in range(3)]
    ip      = 0.006

    fig = plt.figure(figsize=(fig_w, fig_h), dpi=cfg.DPI, facecolor='white')

    col_hdrs = ['Original ROI', 'Integrated Gradients (signed)', 'Grad-CAM++']
    for ci, lab in enumerate(col_hdrs):
        fig.text(col_l[ci] + col_w / 2, top_r0 + f_hdr / 2,
                  lab, ha='center', va='center',
                  fontsize=_FONT_COLHDR, fontweight='normal', color='#222222')

    fig.text(0.5, 1.0 - f_meta * 0.28,
              f"Integrated Gradients + Grad-CAM++  —  {figure_tag}",
              ha='center', va='center',
              fontsize=_FONT_TITLE, fontweight='bold', color='#111111')
    fig.text(0.5, 1.0 - f_meta * 0.76,
              _meta_line(result),
              ha='center', va='center',
              fontsize=_FONT_BODY, fontweight='normal', color=bc)

    fig.text(lr_mar * 0.38, (top_r0 + bot_r0) / 2,
              'Raw Maps', ha='center', va='center', rotation=90,
              fontsize=_FONT_BODY - 0.5, color='#777777', style='italic')
    fig.text(lr_mar * 0.38, (bot_cb + bot_r1) / 2,
              'Overlays', ha='center', va='center', rotation=90,
              fontsize=_FONT_BODY - 0.5, color='#777777', style='italic')

    sub_labels = ['', 'Pixel-level attribution', 'Spatial localization']
    for ci, slbl in enumerate(sub_labels):
        if slbl:
            fig.text(col_l[ci] + col_w / 2, bot_r1 - f_sublbl / 2,
                      slbl, ha='center', va='center',
                      fontsize=_FONT_SUBLBL, color='#555555', style='italic')

    def _ax(left, bottom, width, height, hide_spines=False):
        a = fig.add_axes([left, bottom, width, height])
        a.set_xticks([]); a.set_yticks([])
        for sp in a.spines.values():
            if hide_spines:
                sp.set_visible(False)
            else:
                sp.set_linewidth(0.4); sp.set_color('#cccccc')
        return a

    ax00 = _ax(col_l[0], bot_r0, col_w, f_row)
    ax00.imshow(rgb, aspect='auto')
    for sp in ax00.spines.values():
        sp.set_visible(True); sp.set_edgecolor(bc); sp.set_linewidth(1.8)

    ax01 = _ax(col_l[1], bot_r0, col_w, f_row)
    if ig_sc is not None:
        norm_d = TwoSlopeNorm(vmin=-1.0, vcenter=0.0, vmax=1.0)
        im01   = ax01.imshow(ig_sc, cmap=_IG_CMAP, norm=norm_d,
                              interpolation='bilinear', aspect='auto')
        ax_cb1 = fig.add_axes([col_l[1] + ip, bot_r0 - f_cb + ip,
                                col_w - 2*ip, f_cb - 2*ip])
        cb1    = plt.colorbar(im01, cax=ax_cb1, orientation='horizontal')
        cb1.set_label('Attribution Strength', fontsize=_FONT_CB, labelpad=1)
        cb1.ax.tick_params(labelsize=_FONT_CB - 1, length=1.5, pad=1)
        cb1.ax.xaxis.set_major_locator(mticker.FixedLocator([-1, -0.5, 0, 0.5, 1]))
        cb1.ax.set_xticklabels(['-1', '−0.5', '0', '+0.5', '+1'])
    else:
        ax01.text(0.5, 0.5, 'IG N/A', ha='center', va='center',
                   transform=ax01.transAxes, color='#888888', fontsize=_FONT_BODY)

    ax02 = _ax(col_l[2], bot_r0, col_w, f_row)
    if cam is not None:
        im02 = ax02.imshow(cam, cmap=_CAM_CMAP, norm=Normalize(vmin=0, vmax=1),
                            interpolation='bilinear', aspect='auto')
        if conc is not None:
            ax02.text(0.03, 0.97, f'Conc. {conc:.1f}%',
                       transform=ax02.transAxes, fontsize=_FONT_ANNOT,
                       va='top', ha='left', color='white',
                       bbox=dict(boxstyle='round,pad=0.20', fc='#1a1a1a', alpha=0.70, lw=0))
        ax_cb2 = fig.add_axes([col_l[2] + ip, bot_r0 - f_cb + ip,
                                col_w - 2*ip, f_cb - 2*ip])
        cb2    = plt.colorbar(im02, cax=ax_cb2, orientation='horizontal')
        cb2.set_label('Activation Intensity', fontsize=_FONT_CB, labelpad=1)
        cb2.ax.tick_params(labelsize=_FONT_CB - 1, length=1.5, pad=1)
        cb2.ax.xaxis.set_major_locator(mticker.FixedLocator([0, 0.25, 0.5, 0.75, 1.0]))
        cb2.ax.set_xticklabels(['0', '0.25', '0.5', '0.75', '1'])
    else:
        ax02.text(0.5, 0.5, 'Grad-CAM++ N/A', ha='center', va='center',
                   transform=ax02.transAxes, color='#888888', fontsize=_FONT_BODY)

    ax10 = _ax(col_l[0], bot_r1, col_w, f_row)
    ax10.imshow(rgb, aspect='auto')
    for sp in ax10.spines.values():
        sp.set_linewidth(0.4); sp.set_color('#cccccc')

    ax11 = _ax(col_l[1], bot_r1, col_w, f_row)
    if ig_abs is not None:
        ov_ig = _blend(rgb, ig_abs, cmap=_CAM_CMAP, alpha=alpha)
        ax11.imshow(ov_ig, aspect='auto')
        ax11.text(0.03, 0.97, f'α={alpha:.2f}', transform=ax11.transAxes,
                   fontsize=_FONT_ANNOT, va='top', ha='left', color='white',
                   bbox=dict(boxstyle='round,pad=0.20', fc='#1a1a1a', alpha=0.70, lw=0))
    else:
        ax11.text(0.5, 0.5, 'N/A', ha='center', va='center',
                   transform=ax11.transAxes, color='#888888', fontsize=_FONT_BODY)

    ax12 = _ax(col_l[2], bot_r1, col_w, f_row)
    if cam is not None:
        ov_cam = _blend(rgb, cam, cmap=_CAM_CMAP, alpha=alpha)
        ax12.imshow(ov_cam, aspect='auto')
        if mask is not None:
            ax12.contour(mask.astype(np.float32), levels=[0.5],
                          colors=['#FFD700'], linewidths=0.8, alpha=0.95)
        if conc is not None:
            ax12.text(0.03, 0.97, f'Top-25% region\n{conc:.1f}% activation',
                       transform=ax12.transAxes, fontsize=_FONT_ANNOT,
                       va='top', ha='left', color='white',
                       bbox=dict(boxstyle='round,pad=0.20', fc='#1a1a1a', alpha=0.70, lw=0))
        ax12.text(0.97, 0.03, '— top-25% contour', transform=ax12.transAxes,
                   fontsize=_FONT_ANNOT - 0.5, va='bottom', ha='right', color='#FFD700')
    else:
        ax12.text(0.5, 0.5, 'N/A', ha='center', va='center',
                   transform=ax12.transAxes, color='#888888', fontsize=_FONT_BODY)

    os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
    fig.savefig(save_path, dpi=cfg.DPI, bbox_inches='tight',
                facecolor='white', format=cfg.FMT)
    plt.close(fig)


# ============================================================================
# 15.  FIVE PUBLICATION FIGURES
# ============================================================================

def plot_five_publication_figures(results    : list,
                                   save_dir   : str,
                                   smooth_sig : float = 1.0,
                                   alpha      : float = 0.42) -> None:
    os.makedirs(save_dir, exist_ok=True)
    ex = _select_exemplars(results)

    specs = [
        ('mal_correct',
         'Grid 1 — Best Malignant  (Correct, Highest P(Mal))',
         'fig1_malignant_correct.png'),
        ('ben_correct',
         'Grid 2 — Best Benign  (Correct, Lowest P(Mal))',
         'fig2_benign_correct.png'),
        ('confident_wrong',
         'Grid 3 — Most Confident Misclassification',
         'fig3_confident_misclass.png'),
        ('borderline',
         'Grid 4 — Borderline Case  (P(Mal) ≈ 0.50)',
         'fig4_borderline_case.png'),
        ('random_rep',
         'Grid 5 — Random Representative Sample',
         'fig5_random_representative.png'),
    ]

    print(f"\n  Generating 5 publication figures → {save_dir}/")
    for key, tag, fname in specs:
        r = ex.get(key)
        if r is None:
            print(f"  ⚠️  No sample available for '{key}' — {fname} skipped.")
            continue
        path = os.path.join(save_dir, fname)
        _render_figure(r, figure_tag=tag, save_path=path,
                        smooth_sig=smooth_sig, alpha=alpha)
        sym, _ = _verdict(r)
        print(f"  📊 {fname:<42}  GT={_lstr(r['label'])}  "
              f"Pred={_lstr(r['pred_class'])}  P={r['pred_prob']:.3f}  {sym}")

    print(f"\n  ✅ All figures saved at {cfg.DPI} DPI  ({cfg.FMT.upper()})")
    _print_caption()


def _print_caption() -> None:
    print(f"\n  ─── FIGURE CAPTION (copy into manuscript) ───────────────────")
    print(f"  Explainability analysis of the ConvNeXt-Tiny-SE binary tumour")
    print(f"  classifier using Integrated Gradients (IG) and Grad-CAM++.")
    print(f"  ─────────────────────────────────────────────────────────────")


# ============================================================================
# 16.  COMBINED 3-CASE PUBLICATION FIGURE
# ============================================================================

def plot_combined_publication_figure(results   : list,
                                      save_dir  : str,
                                      smooth_sig: float = 1.0,
                                      alpha     : float = 0.42) -> None:
    os.makedirs(save_dir, exist_ok=True)
    ex = _select_exemplars(results)

    case_specs = [
        ('mal_correct',     'Malignant — Correct',              '#1a7a3c'),
        ('ben_correct',     'Benign — Correct',                  '#1a7a3c'),
        ('confident_wrong', 'Confident Misclassification',       '#c0392b'),
    ]

    cases = []
    for key, label, colour in case_specs:
        r = ex.get(key)
        if r is None:
            print(f"  ⚠️  No sample for '{key}' — combined figure may be incomplete.")
        cases.append((r, label, colour))

    cell     = 2.60
    cb_h     = 0.16
    clbl_h   = 0.28
    div_h    = 0.12
    top_h    = 0.44
    sublbl_h = 0.22

    n_cases   = 3
    n_divs    = n_cases - 1

    fig_w = cell * 3
    fig_h = (top_h + n_cases * (clbl_h + 2 * cell + cb_h)
             + n_divs * div_h + sublbl_h)

    f_top    = top_h    / fig_h
    f_clbl   = clbl_h   / fig_h
    f_cell   = cell     / fig_h
    f_cb     = cb_h     / fig_h
    f_div    = div_h    / fig_h
    f_sublbl = sublbl_h / fig_h

    lr_mar   = 0.013
    col_w    = (1.0 - 2 * lr_mar) / 3
    col_l    = [lr_mar + i * col_w for i in range(3)]
    cb_pad   = f_cb * 0.06

    fig = plt.figure(figsize=(fig_w, fig_h), dpi=cfg.DPI, facecolor='white')

    fig.text(0.5, 1.0 - f_top * 0.40, 'Figure X: Explainability Analysis',
              ha='center', va='center',
              fontsize=_FONT_TITLE, fontweight='bold', color='#111111')

    col_hdrs = ['Original ROI', 'Integrated Gradients (signed)', 'Grad-CAM++']
    for ci, lbl in enumerate(col_hdrs):
        fig.text(col_l[ci] + col_w / 2, 1.0 - f_top * 0.70,
                  lbl, ha='center', va='center',
                  fontsize=_FONT_COLHDR, fontweight='normal', color='#222222')

    case_tops = []
    cursor = 1.0 - f_top
    for ci in range(n_cases):
        case_tops.append(cursor)
        cursor -= (f_clbl + 2 * f_cell + f_cb)
        if ci < n_cases - 1:
            cursor -= f_div

    def _ax(left, bottom, w, h, hide=False):
        a = fig.add_axes([left, bottom, w, h])
        a.set_xticks([]); a.set_yticks([])
        for sp in a.spines.values():
            sp.set_visible(not hide)
            if not hide:
                sp.set_linewidth(0.35); sp.set_color('#cccccc')
        return a

    for ci, ((r, case_lbl, case_col), top) in enumerate(zip(cases, case_tops)):
        fig.text(0.5, top - f_clbl / 2, case_lbl,
                  ha='center', va='center',
                  fontsize=_FONT_BODY + 0.5, fontweight='bold', color=case_col)

        if ci > 0:
            line_y = top + f_div / 2
            fig.add_artist(
                plt.Line2D([lr_mar, 1 - lr_mar], [line_y, line_y],
                           transform=fig.transFigure,
                           color='#dddddd', linewidth=0.6, linestyle='--'))

        if r is None:
            continue

        rgb    = to_float_rgb(r['rgb_u8'])
        ig_raw = r.get('ig_signed_mal')
        cam    = r.get('gpp_cam_mal')
        conc   = r.get('gpp_conc_mal')
        mask   = r.get('gpp_mask_mal')
        sym, bc = _verdict(r)

        ig_sc  = (_norm_signed(ig_raw, smooth_sigma=smooth_sig) if ig_raw is not None else None)
        ig_abs = (_norm_abs(ig_raw,    smooth_sigma=smooth_sig) if ig_raw is not None else None)

        meta = (f"GT: {_lstr(r['label'])}   Pred: {_lstr(r['pred_class'])}   "
                f"{_fmt_prob(r['pred_prob'])}   {sym}")
        fig.text(0.5, top - f_clbl * 0.78, meta,
                  ha='center', va='center', fontsize=_FONT_ANNOT, color=bc)

        top_subA  = top  - f_clbl
        bot_subA  = top_subA - f_cell
        bot_cb    = bot_subA - f_cb
        bot_subB  = bot_cb   - f_cell

        fig.text(lr_mar * 0.38, (top_subA + bot_subA) / 2, 'Raw',
                  ha='center', va='center', rotation=90,
                  fontsize=_FONT_ANNOT - 0.5, color='#888888', style='italic')
        fig.text(lr_mar * 0.38, (bot_cb + bot_subB) / 2, 'Overlay',
                  ha='center', va='center', rotation=90,
                  fontsize=_FONT_ANNOT - 0.5, color='#888888', style='italic')

        ax_A0 = _ax(col_l[0], bot_subA, col_w, f_cell)
        ax_A0.imshow(rgb, aspect='auto')
        for sp in ax_A0.spines.values():
            sp.set_edgecolor(bc); sp.set_linewidth(1.6)

        ax_A1 = _ax(col_l[1], bot_subA, col_w, f_cell)
        if ig_sc is not None:
            norm_d = TwoSlopeNorm(vmin=-1.0, vcenter=0.0, vmax=1.0)
            im_ig  = ax_A1.imshow(ig_sc, cmap=_IG_CMAP, norm=norm_d,
                                   interpolation='bilinear', aspect='auto')
            ax_cb1 = fig.add_axes([col_l[1] + cb_pad, bot_subA - f_cb + cb_pad,
                                    col_w - 2*cb_pad, f_cb - 2*cb_pad])
            cb1 = plt.colorbar(im_ig, cax=ax_cb1, orientation='horizontal')
            cb1.set_label('Attribution Strength', fontsize=_FONT_CB, labelpad=1)
            cb1.ax.tick_params(labelsize=_FONT_CB - 1, length=1.5, pad=1)
            cb1.ax.xaxis.set_major_locator(mticker.FixedLocator([-1, -0.5, 0, 0.5, 1]))
            cb1.ax.set_xticklabels(['-1', '−0.5', '0', '+0.5', '+1'])
            if ci == 0:
                ax_cb1.text(1.01, 0.5, '← shared scale',
                             transform=ax_cb1.transAxes,
                             fontsize=_FONT_ANNOT - 1.5, va='center',
                             color='#888888', style='italic')
        else:
            ax_A1.text(0.5, 0.5, 'IG N/A', ha='center', va='center',
                        transform=ax_A1.transAxes, color='#888888', fontsize=_FONT_BODY)

        ax_A2 = _ax(col_l[2], bot_subA, col_w, f_cell)
        if cam is not None:
            im_cam = ax_A2.imshow(cam, cmap=_CAM_CMAP, norm=Normalize(vmin=0, vmax=1),
                                   interpolation='bilinear', aspect='auto')
            if conc is not None:
                ax_A2.text(0.03, 0.97, f'Conc. {conc:.1f}%',
                            transform=ax_A2.transAxes, fontsize=_FONT_ANNOT - 1,
                            va='top', ha='left', color='white',
                            bbox=dict(boxstyle='round,pad=0.18', fc='#1a1a1a', alpha=0.70, lw=0))
            ax_cb2 = fig.add_axes([col_l[2] + cb_pad, bot_subA - f_cb + cb_pad,
                                    col_w - 2*cb_pad, f_cb - 2*cb_pad])
            cb2 = plt.colorbar(im_cam, cax=ax_cb2, orientation='horizontal')
            cb2.set_label('Activation Intensity', fontsize=_FONT_CB, labelpad=1)
            cb2.ax.tick_params(labelsize=_FONT_CB - 1, length=1.5, pad=1)
            cb2.ax.xaxis.set_major_locator(mticker.FixedLocator([0, 0.25, 0.5, 0.75, 1.0]))
            cb2.ax.set_xticklabels(['0', '0.25', '0.5', '0.75', '1'])
        else:
            ax_A2.text(0.5, 0.5, 'CAM N/A', ha='center', va='center',
                        transform=ax_A2.transAxes, color='#888888', fontsize=_FONT_BODY)

        ax_B0 = _ax(col_l[0], bot_subB, col_w, f_cell)
        ax_B0.imshow(rgb, aspect='auto')
        for sp in ax_B0.spines.values():
            sp.set_linewidth(0.35); sp.set_color('#cccccc')

        ax_B1 = _ax(col_l[1], bot_subB, col_w, f_cell)
        if ig_abs is not None:
            ov_ig = _blend(rgb, ig_abs, cmap=_CAM_CMAP, alpha=alpha)
            ax_B1.imshow(ov_ig, aspect='auto')
            ax_B1.text(0.03, 0.97, f'α={alpha:.2f}', transform=ax_B1.transAxes,
                        fontsize=_FONT_ANNOT - 1, va='top', ha='left', color='white',
                        bbox=dict(boxstyle='round,pad=0.18', fc='#1a1a1a', alpha=0.70, lw=0))
        else:
            ax_B1.text(0.5, 0.5, 'N/A', ha='center', va='center',
                        transform=ax_B1.transAxes, color='#888888', fontsize=_FONT_BODY)

        ax_B2 = _ax(col_l[2], bot_subB, col_w, f_cell)
        if cam is not None:
            ov_cam = _blend(rgb, cam, cmap=_CAM_CMAP, alpha=alpha)
            ax_B2.imshow(ov_cam, aspect='auto')
            if mask is not None:
                ax_B2.contour(mask.astype(np.float32), levels=[0.5],
                               colors=['#FFD700'], linewidths=0.8, alpha=0.95)
            if conc is not None:
                ax_B2.text(0.03, 0.97, f'Top-25%\n{conc:.1f}% activation',
                            transform=ax_B2.transAxes, fontsize=_FONT_ANNOT - 1,
                            va='top', ha='left', color='white',
                            bbox=dict(boxstyle='round,pad=0.18', fc='#1a1a1a', alpha=0.70, lw=0))
            ax_B2.text(0.97, 0.03, '— top-25% contour', transform=ax_B2.transAxes,
                        fontsize=_FONT_ANNOT - 1.5, va='bottom', ha='right', color='#FFD700')
        else:
            ax_B2.text(0.5, 0.5, 'N/A', ha='center', va='center',
                        transform=ax_B2.transAxes, color='#888888', fontsize=_FONT_BODY)

    sub_labels = {1: 'Pixel-level attribution', 2: 'Spatial localization'}
    for ci, slbl in sub_labels.items():
        fig.text(col_l[ci] + col_w / 2, f_sublbl / 2, slbl,
                  ha='center', va='center',
                  fontsize=_FONT_SUBLBL, color='#555555', style='italic')

    out_path = os.path.join(save_dir, 'fig_combined_explainability.png')
    fig.savefig(out_path, dpi=cfg.DPI, bbox_inches='tight',
                facecolor='white', format=cfg.FMT)
    plt.close(fig)
    print(f"  📊 fig_combined_explainability.png  (3-case combined, {cfg.DPI} DPI)")


# ============================================================================
# COMPACT PUBLICATION FIGURE  — v5  (FIX-8: no text overlap)
#
# Root causes of overlap in previous versions
# ─────────────────────────────────────────────
#   O1  ax.set_title() on image axes collided with column headers above.
#       FIX: Removed set_title(). Per-row metadata drawn in dedicated
#            meta_h strips (fig.text) placed ABOVE each image row.
#   O2  Colorbar "Red=…/Blue=…" legend text at cb_bot + f_cb*0.12 sat
#       inside the image cell of the next row.
#       FIX: Moved legend into bottom half of cb strip; made shorter.
#   O3  Row image cells abutted each other with zero gap.
#       FIX: Explicit row_gap between each (meta+cell) block.
#   O4  Colorbar axes height could approach zero for deep figures.
#       FIX: cb_pad expressed as fraction of f_cb (always safe).
#
# New geometry (per row, top → bottom inside its block)
# ──────────────────────────────────────────────────────
#   [meta_h]   row-label + GT/Pred/Prob/Verdict text  (fig.text only)
#   [cell]     3 × image axes
#   — then after row-0 only —
#   [cb_h]     colorbar axes + short legend
#   [row_gap]  blank separator before next row
#
# Total height (3 rows):
#   top_h + 3*(meta_h + cell) + cb_h + 2*row_gap + sublbl_h
#
# All text drawn with fig.text() at exact fractional y positions.
# No ax.set_title(), no ax.text() for per-row headers.
# In-panel annotations (Conc %, α, contour note) remain on image axes
# but are small, opaque-backed, corner-anchored — they never overlap
# the strip text.
# ============================================================================

_COMPACT_ALPHA = 0.45


def _norm_ig_fixed(attr: np.ndarray, smooth_sigma: float = 1.5) -> np.ndarray:
    """Signed IG → (H,W) ∈ [−1,+1], 98th-pct clip, Gaussian post-smooth."""
    scalar = attr.sum(axis=0)
    if smooth_sigma > 0:
        scalar = gaussian_filter(scalar, sigma=smooth_sigma)
    vmax = np.percentile(np.abs(scalar), 98)
    if vmax < 1e-8:
        return np.zeros_like(scalar, dtype=np.float32)
    return np.clip(scalar / vmax, -1.0, 1.0).astype(np.float32)


def _compact_figure_geometry(n_rows: int) -> dict:
    """
    Return geometry dict for an n-row compact figure.

    Layout per row (top → bottom):
      meta_h : metadata text strip  (fig.text only — NO axes inside)
      cell   : image axes row

    After row-0 only:
      cb_h   : colorbar strip

    Between rows (rows 1..n-1 only):
      row_gap : blank separator

    Bottom:
      sublbl_h : sub-label strip
    """
    cell     = 2.70    # image cell (inch, square)
    cb_h     = 0.36    # colorbar strip: bar(38%) + legend(22%) + padding
    meta_h   = 0.28    # NEW: per-row metadata strip above image cells
    top_h    = 0.46    # title + col-header strip
    sublbl_h = 0.20    # sub-labels at bottom
    row_gap  = 0.14    # blank gap between row blocks

    n_gaps   = max(n_rows - 1, 0)

    fig_w    = cell * 3
    fig_h    = (top_h
                + n_rows * (meta_h + cell)
                + cb_h
                + n_gaps * row_gap
                + sublbl_h)

    f        = lambda x: x / fig_h

    # Left margin wide enough for row labels
    lr_mar   = 0.058
    col_w    = (1.0 - lr_mar - 0.008) / 3
    col_l    = [lr_mar + i * col_w for i in range(3)]

    # cb_pad: safe fraction so colorbar height is always positive
    cb_pad   = max(f(cb_h) * 0.07, 0.0008)

    return dict(
        cell=cell, cb_h=cb_h, meta_h=meta_h,
        top_h=top_h, sublbl_h=sublbl_h, row_gap=row_gap,
        fig_w=fig_w, fig_h=fig_h,
        f_top=f(top_h), f_cell=f(cell), f_cb=f(cb_h),
        f_meta=f(meta_h), f_gap=f(row_gap), f_sublbl=f(sublbl_h),
        lr_mar=lr_mar, col_w=col_w, col_l=col_l, cb_pad=cb_pad,
    )


def _compute_compact_row_positions(g: dict, n_rows: int) -> tuple:
    """
    Compute (cell_bot, meta_bot) for each row, from bottom to top.

    Returns (row_positions, cb_pos) where:
      row_positions : list of dicts [{row, cell_bot, meta_bot}, …] ordered row-0 first
      cb_pos        : dict {cb_bot} or None

    For n_rows == 1:
      Layout (bottom → top): sublbl_h | cb_h | cell | meta_h | top_h
      CB is placed BELOW the image cell (above sublbl).

    For n_rows >= 2:
      Layout (bottom → top):
        sublbl_h
        row (n-1): cell + meta_h
        row_gap
        ...
        row 1: cell + meta_h
        cb_h     ← between row-1 and row-0
        row_gap
        row 0: cell + meta_h
        top_h
    """
    f_sub  = g['f_sublbl']
    f_meta = g['f_meta']
    f_cell = g['f_cell']
    f_cb   = g['f_cb']
    f_gap  = g['f_gap']

    if n_rows == 1:
        # Single-row: CB sits between sublbl and image cell
        cb_bot_val = f_sub
        cell_bot   = cb_bot_val + f_cb
        meta_bot   = cell_bot + f_cell
        row_positions = [{'row': 0, 'cell_bot': cell_bot, 'meta_bot': meta_bot}]
        cb_pos        = {'row': 'cb', 'cb_bot': cb_bot_val}
        return row_positions, cb_pos

    # Multi-row: build bottom-up
    positions = []
    cursor    = f_sub

    for ri in range(n_rows - 1, -1, -1):
        cell_bot = cursor
        meta_bot = cursor + f_cell
        positions.append({'row': ri, 'cell_bot': cell_bot, 'meta_bot': meta_bot})
        cursor = meta_bot + f_meta

        if ri == 1:
            # CB placed between row-1 and row-0
            cb_bot_val = cursor
            positions.append({'row': 'cb', 'cb_bot': cb_bot_val})
            cursor += f_cb

        if ri > 0:
            cursor += f_gap  # gap between row blocks

    row_positions = [p for p in reversed(positions) if isinstance(p.get('row'), int)]
    cb_pos        = next((p for p in positions if p.get('row') == 'cb'), None)

    return row_positions, cb_pos


def _draw_compact_header(fig, g: dict, title_str: str) -> None:
    """Suptitle + column headers — all via fig.text()."""
    col_l = g['col_l']; col_w = g['col_w']; f_top = g['f_top']
    fig.text(0.5, 1.0 - f_top * 0.30,
              title_str,
              ha='center', va='center',
              fontsize=_FONT_TITLE, fontweight='bold', color='#111111',
              wrap=False)
    for ci, lbl in enumerate(['Original ROI',
                               'Integrated Gradients (signed)',
                               'Grad-CAM++ (overlay)']):
        fig.text(col_l[ci] + col_w / 2, 1.0 - f_top * 0.78,
                  lbl, ha='center', va='center',
                  fontsize=_FONT_COLHDR, fontweight='normal', color='#333333')


def _draw_compact_sublabels(fig, g: dict) -> None:
    """Bottom sub-labels below column 1 and 2."""
    col_l = g['col_l']; col_w = g['col_w']; f_sublbl = g['f_sublbl']
    for ci, slbl in [(1, 'Pixel-level attribution'),
                     (2, 'Spatial localization')]:
        fig.text(col_l[ci] + col_w / 2, f_sublbl / 2,
                  slbl, ha='center', va='center',
                  fontsize=_FONT_SUBLBL, color='#555555', style='italic')


def _attach_compact_colorbars(fig, im_ig, im_cam,
                               col_l: list, col_w: float,
                               cb_pad: float, cb_bot: float,
                               f_cb: float) -> None:
    """
    Colorbars in the dedicated cb strip. No tick values shown.

    Strip layout (bottom → top):
      [cb_bot]
        0–18%   bottom padding
        18–28%  IG legend text  (fig.text — never touches axes)
        28–40%  gap
        40–78%  colorbar bar axes
        78–88%  label text  (fig.text ABOVE bar — no set_label)
        88–100% top padding
      [cb_bot + f_cb]

    All text uses fig.text() at exact fractional y — no set_label(),
    so nothing can be pushed outside the strip by matplotlib layout.
    """
    bar_h = f_cb * 0.38   # bar height
    bar_b = cb_bot + f_cb * 0.40   # bar bottom (lower-centre of strip)

    # ── IG colorbar (col 1) ──────────────────────────────────────────────────
    if im_ig is not None:
        ax1 = fig.add_axes([col_l[1] + cb_pad,
                             bar_b,
                             col_w - 2 * cb_pad,
                             bar_h])
        cb1 = plt.colorbar(im_ig, cax=ax1, orientation='horizontal')
        cb1.ax.tick_params(length=0, pad=0, labelsize=0)
        cb1.ax.set_xticklabels([])

        # Label — placed via fig.text ABOVE the bar (never overlaps legend below)
        fig.text(
            col_l[1] + col_w / 2,
            cb_bot + f_cb * 0.95,
            'Attribution Strength',
            ha='center', va='center',
            fontsize=_FONT_CB,
            color='#222222',
        )

        # Legend — placed via fig.text in the lower portion of the strip
        fig.text(
            col_l[1] + col_w / 2,
            cb_bot + f_cb * 0.23,
            'Red: positive contribution toward malignant class'
            '      '
            'Blue: negative contribution',
            ha='center', va='center',
            fontsize=_FONT_ANNOT - 1.5,
            color='#444444',
            style='italic',
        )

    # ── Grad-CAM++ colorbar (col 2) ──────────────────────────────────────────
    if im_cam is not None:
        ax2 = fig.add_axes([col_l[2] + cb_pad,
                             bar_b,
                             col_w - 2 * cb_pad,
                             bar_h])
        cb2 = plt.colorbar(im_cam, cax=ax2, orientation='horizontal')
        cb2.ax.tick_params(length=0, pad=0, labelsize=0)
        cb2.ax.set_xticklabels([])

        # Label — placed via fig.text ABOVE the bar
        fig.text(
            col_l[2] + col_w / 2,
            cb_bot + f_cb * 0.95,
            'Activation Intensity',
            ha='center', va='center',
            fontsize=_FONT_CB,
            color='#222222',
        )


def _draw_compact_row_metadata(fig, r: dict, meta_bot: float, g: dict) -> None:
    """
    Draw per-row metadata in the meta strip (meta_bot to meta_bot+f_meta).

    All text placed via fig.text() — no axes, no set_title().
    Two lines:
      line 1 (top):    row case label  e.g. "Malignant — Correct"
      line 2 (bottom): GT / Pred / Prob / verdict
    """
    col_l  = g['col_l']
    col_w  = g['col_w']
    f_meta = g['f_meta']
    sym, bc = _verdict(r)

    # Two text lines distributed within the meta strip
    y_top = meta_bot + f_meta * 0.72
    y_bot = meta_bot + f_meta * 0.28

    # Row label shown as left-margin rotated label — just the verdict line centred
    meta_str = (f"GT: {_lstr(r['label'])}     "
                f"Pred: {_lstr(r['pred_class'])}     "
                f"{_fmt_prob(r['pred_prob'])}     {sym}")
    fig.text(0.5, y_bot, meta_str,
              ha='center', va='center',
              fontsize=_FONT_ANNOT, color=bc, fontweight='normal')


def _render_compact_row(fig, r: dict,
                         row_label : str,
                         cell_bot  : float,
                         meta_bot  : float,
                         g         : dict,
                         smooth_sig: float = 1.5,
                         alpha     : float = 0.45) -> tuple:
    """
    Render one row of a compact figure.

    Parameters
    ----------
    cell_bot  : figure-fraction bottom of the image cell
    meta_bot  : figure-fraction bottom of the metadata strip above the cell

    All metadata text → fig.text() in the meta strip (no ax.set_title).
    In-panel annotations remain small, corner-anchored, opaque-backed.

    Returns (im_ig, im_cam) AxesImage objects for colorbar attachment.
    """
    f_cell  = g['f_cell']
    f_meta  = g['f_meta']
    col_l   = g['col_l']
    col_w   = g['col_w']
    lr_mar  = g['lr_mar']

    sym, bc  = _verdict(r)
    rgb      = to_float_rgb(r['rgb_u8'])
    ig_raw   = r.get('ig_signed_mal')
    cam_raw  = r.get('gpp_cam_mal')
    conc     = r.get('gpp_conc_mal')
    mask     = r.get('gpp_mask_mal')
    ig_sc    = _norm_ig_fixed(ig_raw, smooth_sigma=smooth_sig) if ig_raw is not None else None

    # ── Metadata strip text (fig.text — never overlaps axes) ─────────────
    # Row case label (top of meta strip)
    fig.text(0.5, meta_bot + f_meta * 0.75,
              row_label,
              ha='center', va='center',
              fontsize=_FONT_BODY, fontweight='bold', color=bc)

    # Verdict / stats line (bottom of meta strip)
    meta_str = (f"GT: {_lstr(r['label'])}   |   "
                f"Pred: {_lstr(r['pred_class'])}   |   "
                f"{_fmt_prob(r['pred_prob'])}   |   {sym}")
    fig.text(0.5, meta_bot + f_meta * 0.28,
              meta_str,
              ha='center', va='center',
              fontsize=_FONT_ANNOT, color=bc, fontweight='normal')

    # ── Left-margin row index label (small, rotated, outside axes) ────────
    fig.text(lr_mar * 0.35, cell_bot + f_cell / 2,
              row_label.split('—')[0].strip(),
              ha='center', va='center', rotation=90,
              fontsize=_FONT_ANNOT - 1.5,
              color='#777777', style='italic')

    # ── Axis factory ──────────────────────────────────────────────────────
    def _ax(left, bottom, w, h):
        a = fig.add_axes([left, bottom, w, h])
        a.set_xticks([]); a.set_yticks([])
        for sp in a.spines.values():
            sp.set_linewidth(0.35); sp.set_color('#cccccc')
        return a

    # ── Col 0: Original ROI — thin neutral border ─────────────────────────
    ax0 = _ax(col_l[0], cell_bot, col_w, f_cell)
    ax0.imshow(rgb, aspect='auto')
    for sp in ax0.spines.values():
        sp.set_linewidth(0.8); sp.set_color('#999999')

    # ── Col 1: IG signed map ─────────────────────────────────────────────
    ax1   = _ax(col_l[1], cell_bot, col_w, f_cell)
    im_ig = None
    if ig_sc is not None:
        norm_d = TwoSlopeNorm(vmin=-1.0, vcenter=0.0, vmax=1.0)
        im_ig  = ax1.imshow(ig_sc, cmap=_IG_CMAP, norm=norm_d,
                             interpolation='bilinear', aspect='auto')
    else:
        ax1.text(0.5, 0.5, 'IG N/A', ha='center', va='center',
                  transform=ax1.transAxes, color='#888888', fontsize=_FONT_BODY)

    # ── Col 2: Grad-CAM++ overlay + contour ──────────────────────────────
    ax2    = _ax(col_l[2], cell_bot, col_w, f_cell)
    im_cam = None
    if cam_raw is not None:
        ov_cam = _blend(rgb, cam_raw, cmap=_CAM_CMAP, alpha=alpha)
        ax2.imshow(ov_cam, aspect='auto')
        if mask is not None:
            ax2.contour(mask.astype(np.float32), levels=[0.5],
                         colors=['#FFD700'], linewidths=0.7, alpha=0.95)

        # Invisible image for fixed-range colorbar
        im_cam = ax2.imshow(cam_raw, cmap=_CAM_CMAP,
                             norm=Normalize(0, 1), visible=False, aspect='auto')
    else:
        ax2.text(0.5, 0.5, 'CAM N/A', ha='center', va='center',
                  transform=ax2.transAxes, color='#888888', fontsize=_FONT_BODY)

    return im_ig, im_cam


def _select_ranked_exemplars(results: list) -> dict:
    """Return up to 3 ranked candidates per category."""
    correct = [r for r in results if r['label'] == r['pred_class']]
    wrong   = [r for r in results if r['label'] != r['pred_class']]

    mal_sorted = sorted([r for r in correct if r['label'] == 1],
                         key=lambda r: r['pred_prob'], reverse=True)
    ben_sorted = sorted([r for r in correct if r['label'] == 0],
                         key=lambda r: r['pred_prob'])
    mis_sorted = sorted(wrong, key=lambda r: r['confidence'], reverse=True)

    def _pad(lst, n=3):
        if not lst:
            return [None] * n
        while len(lst) < n:
            lst = lst + [lst[-1]]
        return lst[:n]

    return {
        'mal_correct'    : _pad(mal_sorted),
        'ben_correct'    : _pad(ben_sorted),
        'confident_wrong': _pad(mis_sorted),
    }


def _build_compact_combined(results   : list,
                              save_dir  : str,
                              version   : int,
                              smooth_sig: float = 1.5,
                              alpha     : float = 0.45) -> None:
    """Build one 3-row combined compact figure using rank-`version` exemplars."""
    ranked = _select_ranked_exemplars(results)
    idx    = version - 1

    row_specs = [
        (ranked['mal_correct'][idx],     'Malignant \u2014 Correct',         '#1a7a3c'),
        (ranked['ben_correct'][idx],     'Benign \u2014 Correct',             '#1a7a3c'),
        (ranked['confident_wrong'][idx], 'Malignant \u2014 Misclassified',    '#c0392b'),
    ]

    n_rows = 3
    g      = _compact_figure_geometry(n_rows=n_rows)
    fig    = plt.figure(figsize=(g['fig_w'], g['fig_h']), dpi=cfg.DPI, facecolor='white')

    TITLE = ('Pixel-level (Integrated Gradients) and spatial (Grad-CAM++) '
             'explainability analysis.')
    _draw_compact_header(fig, g, TITLE)

    # Compute row positions using the centralised helper
    row_positions, cb_pos = _compute_compact_row_positions(g, n_rows)

    first_im_ig = first_im_cam = None

    for ri, ((r, row_lbl, case_col), pos) in enumerate(
            zip(row_specs, row_positions)):
        if r is None:
            print(f"  ⚠️  v{version} row '{row_lbl}': no sample — skipped.")
            continue

        im_ig, im_cam = _render_compact_row(
            fig, r, row_lbl,
            cell_bot  = pos['cell_bot'],
            meta_bot  = pos['meta_bot'],
            g         = g,
            smooth_sig= smooth_sig,
            alpha     = alpha,
        )
        if ri == 0:
            first_im_ig, first_im_cam = im_ig, im_cam

    # Attach colorbars in the dedicated cb strip
    if cb_pos is not None and (first_im_ig is not None or first_im_cam is not None):
        _attach_compact_colorbars(
            fig, first_im_ig, first_im_cam,
            g['col_l'], g['col_w'],
            g['cb_pad'], cb_pos['cb_bot'], g['f_cb'],
        )

    _draw_compact_sublabels(fig, g)

    fname = f'fig_compact_combined_v{version}.png'
    p = os.path.join(save_dir, fname)
    fig.savefig(p, dpi=cfg.DPI, bbox_inches='tight',
                facecolor='white', format=cfg.FMT)
    plt.close(fig)
    print(f"  📊 {fname}  (rank-{version} exemplars)")


def plot_compact_publication_figure(results   : list,
                                     save_dir  : str,
                                     smooth_sig: float = 1.5,
                                     alpha     : float = 0.45) -> None:
    """
    Generate all compact publication figures.

    Combined (3 versions using rank-1/2/3 exemplars):
      fig_compact_combined_v1.png   ← best samples (rank-1)
      fig_compact_combined_v2.png   ← second-best  (rank-2)
      fig_compact_combined_v3.png   ← third-best   (rank-3)

    Individual single-row (5 best cases, rank-1 exemplar each):
      fig_compact_1_malignant_correct.png
      fig_compact_2_benign_correct.png
      fig_compact_3_misclassified.png
      fig_compact_4_borderline.png
      fig_compact_5_random.png
    """
    os.makedirs(save_dir, exist_ok=True)

    print(f"  Generating 3 combined compact figures …")
    for v in (1, 2, 3):
        _build_compact_combined(results, save_dir, version=v,
                                 smooth_sig=smooth_sig, alpha=alpha)

    ex = _select_exemplars(results)
    TITLE_BASE = ('Pixel-level (Integrated Gradients) and spatial '
                  '(Grad-CAM++) explainability analysis.')

    individual_specs = [
        (ex.get('mal_correct'),
         'Malignant \u2014 Correct',
         'Best malignant correct \u2014 highest P(Mal)',
         'fig_compact_1_malignant_correct.png'),
        (ex.get('ben_correct'),
         'Benign \u2014 Correct',
         'Best benign correct \u2014 lowest P(Mal)',
         'fig_compact_2_benign_correct.png'),
        (ex.get('confident_wrong'),
         'Malignant \u2014 Misclassified',
         'Confident misclassification',
         'fig_compact_3_misclassified.png'),
        (ex.get('borderline'),
         'Borderline \u2014 P(Mal)\u22480.50',
         'Maximum uncertainty \u2014 borderline case',
         'fig_compact_4_borderline.png'),
        (ex.get('random_rep'),
         'Representative Sample',
         'Seeded random representative sample',
         'fig_compact_5_random.png'),
    ]

    print(f"  Generating 5 individual compact figures …")
    for r, row_lbl, subtitle, fname in individual_specs:
        if r is None:
            print(f"  ⚠️  '{row_lbl}' — no sample, {fname} skipped.")
            continue

        n_rows  = 1
        gi      = _compact_figure_geometry(n_rows=n_rows)
        fig_i   = plt.figure(figsize=(gi['fig_w'], gi['fig_h']),
                             dpi=cfg.DPI, facecolor='white')
        full_title = TITLE_BASE + '  ' + subtitle + '.'
        _draw_compact_header(fig_i, gi, full_title)

        row_positions_i, cb_pos_i = _compute_compact_row_positions(gi, n_rows)
        pos_i = row_positions_i[0]

        im_ig, im_cam = _render_compact_row(
            fig_i, r, row_lbl,
            cell_bot  = pos_i['cell_bot'],
            meta_bot  = pos_i['meta_bot'],
            g         = gi,
            smooth_sig= smooth_sig,
            alpha     = alpha,
        )

        if cb_pos_i is not None:
            _attach_compact_colorbars(
                fig_i, im_ig, im_cam,
                gi['col_l'], gi['col_w'],
                gi['cb_pad'], cb_pos_i['cb_bot'], gi['f_cb'],
            )

        _draw_compact_sublabels(fig_i, gi)

        p = os.path.join(save_dir, fname)
        fig_i.savefig(p, dpi=cfg.DPI, bbox_inches='tight',
                      facecolor='white', format=cfg.FMT)
        plt.close(fig_i)
        sym, _ = _verdict(r)
        print(f"  📊 {fname:<45}  GT={_lstr(r['label'])}  "
              f"Pred={_lstr(r['pred_class'])}  {_fmt_prob(r['pred_prob'])}  {sym}")

    _print_compact_caption()


def _print_compact_caption() -> None:
    print(f"\n  ─── COMPACT FIGURE CAPTION (copy into manuscript) ───────────────────")
    print(f"  Pixel-level (Integrated Gradients) and spatial (Grad-CAM++)")
    print(f"  explainability analysis of the ConvNeXt-Tiny-SE binary tumour")
    print(f"  classifier. Columns: (a) original ROI; (b) IG signed attribution")
    print(f"  map—red indicates positive contribution toward the malignant logit,")
    print(f"  blue indicates negative contribution; (c) Grad-CAM++ activation")
    print(f"  overlay (α=0.45) with gold contour (top-25% activation boundary).")
    print(f"  All attributions target class 1 (Malignant). IG baseline: Gaussian-")
    print(f"  blurred input (σ=10 px). Integration: 200 steps (Gauss-Legendre).")
    print(f"  IG post-smoothing: σ=1.5 px; 98th-pct clip; colourscale [−1, +1].")
    print(f"  Grad-CAM++ layer: model.features[-1][-1]; γ=0.7 correction applied.")
    print(f"  ─────────────────────────────────────────────────────────────────────")


# ============================================================================
# 16.  SUPPLEMENTAL GALLERIES
# ============================================================================

def plot_ig_signed_gallery(results   : list,
                            save_dir  : str,
                            max_items : int = 20) -> None:
    valid = [r for r in results if r.get('ig_signed_mal') is not None][:max_items]
    if not valid:
        return

    n    = len(valid)
    rh   = 2.6
    fig  = plt.figure(figsize=(10.5, rh * n + 0.7), dpi=cfg.DPI, facecolor='white')

    for ci, lab in enumerate(['Original', 'IG — Signed Attribution', 'IG — Overlay (α=0.42)']):
        fig.text((ci + 0.5) / 3, 1 - 0.35 / (rh * n + 0.7),
                  lab, ha='center', va='top',
                  fontsize=_FONT_COLHDR, fontweight='normal')

    for ri, r in enumerate(valid):
        rgb    = to_float_rgb(r['rgb_u8'])
        ig_raw = r['ig_signed_mal']
        ig_sc  = _norm_signed(ig_raw, smooth_sigma=cfg.IG_SMOOTH_SIGMA)
        ig_abs = _norm_abs(ig_raw,    smooth_sigma=cfg.IG_SMOOTH_SIGMA)
        lbl    = r['label']; pred = r['pred_class']
        sym, bc = _verdict(r)
        bot    = 1 - (ri + 1) * rh / (rh * n + 0.7)
        h      = rh / (rh * n + 0.7) * 0.93

        for ci in range(3):
            ax = fig.add_axes([ci / 3 + 0.01, bot, 1/3 - 0.02, h])
            ax.set_xticks([]); ax.set_yticks([])

            if ci == 0:
                ax.imshow(rgb, aspect='auto')
                ax.set_title(
                    f"GT:{_lstr(lbl)}  Pred:{_lstr(pred)}  {sym}  P={r['pred_prob']:.3f}",
                    fontsize=6.5, color=bc, pad=3, fontweight='bold')
                for sp in ax.spines.values():
                    sp.set_edgecolor(bc); sp.set_linewidth(1.8)

            elif ci == 1:
                nd = TwoSlopeNorm(vmin=-1.0, vcenter=0.0, vmax=1.0)
                im = ax.imshow(ig_sc, cmap=_IG_CMAP, norm=nd,
                                interpolation='bilinear', aspect='auto')
                d = r.get('ig_delta_mal')
                if d is not None:
                    ax.text(0.03, 0.97, f'Δ={d:.4f}', transform=ax.transAxes,
                             fontsize=6, va='top', color='white',
                             bbox=dict(boxstyle='round,pad=0.2', fc='#1a1a1a', alpha=0.75, lw=0))
                cb = plt.colorbar(im, ax=ax, orientation='vertical', fraction=0.046, pad=0.02)
                cb.set_label('Attribution', fontsize=5.5, labelpad=2)
                cb.ax.tick_params(labelsize=5)

            else:
                ov = _blend(rgb, ig_abs, cmap=_CAM_CMAP, alpha=cfg.ALPHA)
                ax.imshow(ov, aspect='auto')
                for sp in ax.spines.values():
                    sp.set_linewidth(0.3); sp.set_color('#cccccc')

    fig.suptitle(
        "Integrated Gradients — Signed Attribution Gallery\n"
        f"Red=promotes malignant  |  Blue=inhibits malignant  |  "
        f"IG baseline: blurred input (σ={cfg.BLUR_SIGMA:.0f}px)  |  "
        f"Post-smooth σ={cfg.IG_SMOOTH_SIGMA:.1f}px",
        fontsize=_FONT_TITLE, fontweight='bold', y=1.01)

    path = os.path.join(save_dir, 'ig_signed_gallery.png')
    fig.savefig(path, dpi=cfg.DPI, bbox_inches='tight', facecolor='white', format=cfg.FMT)
    plt.close(fig)
    print(f"  📊 Saved: ig_signed_gallery.png")


def plot_gradcam_gallery(results   : list,
                          save_dir  : str,
                          max_items : int = 20) -> None:
    valid = [r for r in results if r.get('gpp_cam_mal') is not None][:max_items]
    if not valid:
        return

    n   = len(valid)
    rh  = 2.6
    fig = plt.figure(figsize=(10.5, rh * n + 0.7), dpi=cfg.DPI, facecolor='white')

    for ci, lab in enumerate(['Original', 'Grad-CAM++ — Heatmap', 'Grad-CAM++ — Overlay + Contour']):
        fig.text((ci + 0.5) / 3, 1 - 0.35 / (rh * n + 0.7),
                  lab, ha='center', va='top', fontsize=_FONT_COLHDR, fontweight='normal')

    for ri, r in enumerate(valid):
        rgb  = to_float_rgb(r['rgb_u8'])
        cam  = r['gpp_cam_mal']
        conc = r.get('gpp_conc_mal')
        mask = r.get('gpp_mask_mal')
        lbl  = r['label']; pred = r['pred_class']
        sym, bc = _verdict(r)
        bot  = 1 - (ri + 1) * rh / (rh * n + 0.7)
        h    = rh / (rh * n + 0.7) * 0.93

        for ci in range(3):
            ax = fig.add_axes([ci / 3 + 0.01, bot, 1/3 - 0.02, h])
            ax.set_xticks([]); ax.set_yticks([])

            if ci == 0:
                ax.imshow(rgb, aspect='auto')
                ax.set_title(
                    f"GT:{_lstr(lbl)}  Pred:{_lstr(pred)}  {sym}  P={r['pred_prob']:.3f}",
                    fontsize=6.5, color=bc, pad=3, fontweight='bold')
                for sp in ax.spines.values():
                    sp.set_edgecolor(bc); sp.set_linewidth(1.8)

            elif ci == 1:
                im = ax.imshow(cam, cmap=_CAM_CMAP, norm=Normalize(0, 1),
                                interpolation='bilinear', aspect='auto')
                if conc is not None:
                    ax.text(0.03, 0.97, f'Conc. {conc:.1f}%', transform=ax.transAxes,
                             fontsize=6, va='top', color='white',
                             bbox=dict(boxstyle='round,pad=0.2', fc='#1a1a1a', alpha=0.75, lw=0))
                cb = plt.colorbar(im, ax=ax, orientation='vertical', fraction=0.046, pad=0.02)
                cb.set_label('Intensity', fontsize=5.5, labelpad=2)
                cb.ax.tick_params(labelsize=5)

            else:
                ov = _blend(rgb, cam, cmap=_CAM_CMAP, alpha=cfg.ALPHA)
                ax.imshow(ov, aspect='auto')
                if mask is not None:
                    ax.contour(mask.astype(np.float32), levels=[0.5],
                                colors=['#FFD700'], linewidths=0.8, alpha=0.95)
                for sp in ax.spines.values():
                    sp.set_linewidth(0.3); sp.set_color('#cccccc')

    fig.suptitle(
        "Grad-CAM++ — Activation Gallery\n"
        "Gold contour = top-25% activation region  |  "
        "Layer: model.features[-1][-1]  |  target: Malignant class",
        fontsize=_FONT_TITLE, fontweight='bold', y=1.01)

    path = os.path.join(save_dir, 'gradcam_gallery.png')
    fig.savefig(path, dpi=cfg.DPI, bbox_inches='tight', facecolor='white', format=cfg.FMT)
    plt.close(fig)
    print(f"  📊 Saved: gradcam_gallery.png")


# ============================================================================
# 17.  SUMMARY JSON / CSV
# ============================================================================

def save_summary(results: list, save_dir: str) -> None:
    rows = []
    for r in results:
        sym, _ = _verdict(r)
        row = dict(
            filename       = Path(r['path']).name,
            ground_truth   = _lstr(r['label']),
            predicted      = _lstr(r['pred_class']),
            prob_malignant = round(r['pred_prob'], 5),
            confidence     = round(r['confidence'], 5),
            correct        = r['label'] == r['pred_class'],
        )
        for tname in ('mal', 'pred'):
            ig_raw = r.get(f'ig_signed_{tname}')
            d      = r.get(f'ig_delta_{tname}')
            cam    = r.get(f'gpp_cam_{tname}')
            conc   = r.get(f'gpp_conc_{tname}')
            if ig_raw is not None:
                ig_abs = _norm_abs(ig_raw)
                row[f'ig_{tname}_mean'] = round(float(ig_abs.mean()), 6)
                row[f'ig_{tname}_max']  = round(float(ig_abs.max()),  6)
            if d is not None:
                row[f'ig_{tname}_delta'] = round(d, 6)
            if cam is not None:
                row[f'gpp_{tname}_mean'] = round(float(cam.mean()), 6)
                row[f'gpp_{tname}_max']  = round(float(cam.max()),  6)
            if conc is not None:
                row[f'gpp_{tname}_conc'] = round(conc, 2)
        rows.append(row)

    jpath = os.path.join(save_dir, 'explainability_summary.json')
    cpath = os.path.join(save_dir, 'explainability_summary.csv')
    with open(jpath, 'w') as f:
        json.dump(rows, f, indent=2)
    pd.DataFrame(rows).to_csv(cpath, index=False)
    print(f"  📄 {jpath}")
    print(f"  📄 {cpath}")


# ============================================================================
# 18.  TEST-SET LOADER
# ============================================================================

_FRCNN = re.compile(r'^(.+?)_box\d+_iou\d+_conf\d+', re.IGNORECASE)
_ROI   = re.compile(r'^(.+?)_roi\d*$', re.IGNORECASE)

_COL_MAP = {
    'roi_filepath': ['roi_filepath','filepath','path','roi_path',
                     'image_path','filename','file'],
    'class'       : ['class','label','class_name','category'],
    'split'       : ['split','subset','partition'],
}


def load_test_df(roi_dir: str) -> pd.DataFrame:
    def _norm(df: pd.DataFrame, default_split: str = 'test') -> pd.DataFrame:
        low = {c.lower(): c for c in df.columns}
        cm  = {}
        for canon, aliases in _COL_MAP.items():
            for a in aliases:
                if a in low: cm[low[a]] = canon; break
        df = df.rename(columns=cm)
        if 'class' not in df.columns:
            df['class'] = df['roi_filepath'].apply(
                lambda p: 'malignant' if 'malignant' in str(p).lower() else 'benign')
        if 'split' not in df.columns:
            df['split'] = default_split
        df['class'] = df['class'].apply(
            lambda v: 'malignant'
            if str(v).strip().lower() in ('malignant','1','mal') else 'benign')

        def _res(row):
            raw = str(row['roi_filepath'])
            if os.path.isabs(raw) and os.path.exists(raw):
                return raw
            fname = Path(raw).name
            for s in [str(row.get('split','')).strip().lower(), 'test','val','train']:
                for c in [str(row.get('class','')).strip().lower(), 'benign','malignant']:
                    p = os.path.join(roi_dir, s, c, fname)
                    if os.path.exists(p):
                        return p
            return os.path.join(roi_dir, 'test', str(row.get('class','benign')).lower(), fname)

        df['roi_filepath'] = df.apply(_res, axis=1)
        return df[['roi_filepath','class','split']].copy()

    m_all  = os.path.join(roi_dir, 'all_splits_roi_manifest.csv')
    m_test = os.path.join(roi_dir, 'test_roi_manifest.csv')

    if os.path.exists(m_all):
        print(f"  📂 {m_all}")
        df = _norm(pd.read_csv(m_all))
        df = df[df['split'] == 'test'].reset_index(drop=True)
    elif os.path.exists(m_test):
        print(f"  📂 {m_test}")
        df = _norm(pd.read_csv(m_test), default_split='test')
    else:
        print("  ⚠️  No manifest found — scanning folder …")
        rows = []
        for cls in ('benign', 'malignant'):
            folder = os.path.join(roi_dir, 'test', cls)
            if not os.path.isdir(folder):
                continue
            for fname in os.listdir(folder):
                if fname.lower().endswith(('.png','.jpg','.jpeg')):
                    rows.append({'roi_filepath': os.path.join(folder, fname),
                                  'class': cls, 'split': 'test'})
        df = pd.DataFrame(rows)

    mask = df['roi_filepath'].apply(os.path.exists)
    if (~mask).sum():
        print(f"  ⚠️  Dropped {(~mask).sum()} missing files.")
    df = df[mask].reset_index(drop=True)

    n_b = (df['class'] == 'benign').sum()
    n_m = (df['class'] == 'malignant').sum()
    print(f"  ✅ Test set: {len(df)} ROIs  (Benign={n_b}, Malignant={n_m})")
    return df


# ============================================================================
# 19.  MAIN PIPELINE
# ============================================================================

def run_pipeline(n_samples: int = 20) -> List[Dict]:
    assert n_samples >= 20, \
        f"n_samples={n_samples} — use ≥20 to populate all 5 figure categories."

    random.seed(cfg.RANDOM_SEED)
    np.random.seed(cfg.RANDOM_SEED)
    torch.manual_seed(cfg.RANDOM_SEED)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\n✅ Device         : {device}")
    if torch.cuda.is_available():
        print(f"✅ GPU            : {torch.cuda.get_device_name(0)}")
    print(f"✅ Captum         : {CAPTUM_OK}")
    print(f"✅ Grad-CAM++     : {GRADCAM_OK}")
    print(f"✅ N_IG_STEPS     : {cfg.N_IG_STEPS}")
    print(f"✅ IG baseline    : '{cfg.IG_BASELINE}' (σ={cfg.BLUR_SIGMA} px)")
    print(f"✅ IG smoothing   : σ={cfg.IG_SMOOTH_SIGMA} px")
    print(f"✅ Overlay alpha  : {cfg.ALPHA}")

    os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
    per_sample_dir = os.path.join(cfg.OUTPUT_DIR, 'per_sample')
    os.makedirs(per_sample_dir, exist_ok=True)
    pub_dir = os.path.join(cfg.OUTPUT_DIR, 'publication_figures')
    os.makedirs(pub_dir, exist_ok=True)

    model = load_model(cfg.CHECKPOINT_PATH, device)
    assert not model.training

    print(f"\n{'='*60}\n  LOADING TEST SET\n{'='*60}")
    test_df = load_test_df(cfg.ROI_DATASET_DIR)

    mal_df = test_df[test_df['class'] == 'malignant']
    ben_df = test_df[test_df['class'] == 'benign']
    n_each = n_samples // 2

    sel = pd.concat([
        mal_df.sample(min(n_each, len(mal_df)), random_state=cfg.RANDOM_SEED),
        ben_df.sample(min(n_each, len(ben_df)), random_state=cfg.RANDOM_SEED),
    ]).sample(frac=1, random_state=cfg.RANDOM_SEED).reset_index(drop=True)

    n_sel = len(sel)
    print(f"\n  Selected {n_sel} ROIs  "
          f"(Mal={(sel['class']=='malignant').sum()}, "
          f"Ben={(sel['class']=='benign').sum()})")

    print(f"\n{'='*60}\n  ATTRIBUTION  [IG + Grad-CAM++]\n{'='*60}")
    results = []

    for si, (_, row) in enumerate(sel.iterrows(), 1):
        path  = str(row['roi_filepath'])
        label = cfg.C2L.get(str(row['class']).lower(), -1)
        print(f"\n  [{si:02d}/{n_sel}]  {Path(path).name}  (GT: {_lstr(label)})")

        r = explain_one(model, path, label, device)
        if r is None:
            continue

        sym, _ = _verdict(r)
        print(f"       Pred={_lstr(r['pred_class'])} {sym}  "
              f"{_fmt_prob(r['pred_prob'])}  conf={r['confidence']:.3f}")

        safe  = re.sub(r'[^\w\-.]', '_', Path(path).name)
        fpath = os.path.join(per_sample_dir, f'{si:04d}_{safe}.png')
        _render_figure(r,
                        figure_tag = f"Sample {si:02d}  |  {_lstr(label)}",
                        save_path  = fpath,
                        smooth_sig = cfg.IG_SMOOTH_SIGMA,
                        alpha      = cfg.ALPHA)
        print(f"       💾 {Path(fpath).name}")
        results.append(r)

    if not results:
        print("  ❌ No results produced.")
        return results

    print(f"\n{'='*60}\n  PUBLICATION FIGURES\n{'='*60}")
    plot_five_publication_figures(results, save_dir=pub_dir,
                                   smooth_sig=cfg.IG_SMOOTH_SIGMA, alpha=cfg.ALPHA)

    print(f"\n{'='*60}\n  COMBINED PUBLICATION FIGURE\n{'='*60}")
    plot_combined_publication_figure(results, save_dir=pub_dir,
                                      smooth_sig=cfg.IG_SMOOTH_SIGMA, alpha=cfg.ALPHA)

    print(f"\n{'='*60}\n  COMPACT PUBLICATION FIGURE\n{'='*60}")
    plot_compact_publication_figure(results, save_dir=pub_dir,
                                     smooth_sig=cfg.IG_SMOOTH_SIGMA, alpha=cfg.ALPHA)

    print(f"\n{'='*60}\n  SUPPLEMENTAL GALLERIES\n{'='*60}")
    gallery_dir = os.path.join(cfg.OUTPUT_DIR, 'galleries')
    os.makedirs(gallery_dir, exist_ok=True)
    plot_ig_signed_gallery(results,  gallery_dir, max_items=n_samples)
    plot_gradcam_gallery(results,    gallery_dir, max_items=n_samples)

    save_summary(results, cfg.OUTPUT_DIR)

    n_ok    = sum(r['label'] == r['pred_class'] for r in results)
    concs   = [r['gpp_conc_mal'] for r in results if r.get('gpp_conc_mal')]
    deltas  = [r['ig_delta_mal'] for r in results if r.get('ig_delta_mal')]

    print(f"\n{'='*60}\n  🎉  PIPELINE COMPLETE\n{'='*60}")
    print(f"  Samples explained       : {len(results)}")
    print(f"  Correct / Wrong         : {n_ok} / {len(results) - n_ok}")
    if concs:
        print(f"  Grad-CAM++ conc (mean)  : {np.mean(concs):.1f}% ± {np.std(concs):.1f}%")
    if deltas:
        print(f"  IG convergence Δ (mean) : {np.mean(deltas):.5f}  (max={np.max(deltas):.5f})")
    print(f"\n  Output directory: {cfg.OUTPUT_DIR}/")
    print(f"{'='*60}")
    return results


# ============================================================================
# 20.  ENTRY POINT
# ============================================================================

if __name__ == '__main__':
    run_pipeline(n_samples=20)


✅ Device         : cuda
✅ GPU            : Tesla T4
✅ Captum         : True
✅ Grad-CAM++     : True
✅ N_IG_STEPS     : 100
✅ IG baseline    : 'blur' (σ=10.0 px)
✅ IG smoothing   : σ=1.0 px
✅ Overlay alpha  : 0.4

  LOADING CHECKPOINT
  /kaggle/working/results_stage3c_cv/final_model/final_best.pth
  ✅ epoch=24  best_metric=0.9274  params=28.1M
  ✅ missing=0  unexpected=0
  ✅ Grad-CAM++ layer: model.features[-1][-1] → CNBlock


  LOADING TEST SET
  ⚠️  No manifest found — scanning folder …
  ✅ Test set: 283 ROIs  (Benign=242, Malignant=41)

  Selected 20 ROIs  (Mal=10, Ben=10)

  ATTRIBUTION  [IG + Grad-CAM++]

  [01/20]  IMG000027_roi0_osteosarcoma_ok_iou0.83_conf0.959.jpg  (GT: Malignant)
     IG + Grad-CAM++  target='mal' (class 1)
       IG  Δ=0.00037
       Grad-CAM++  conc=59.3%
       Pred=Malignant ✓  P(Mal)=0.844  conf=0.844
       💾 0001_IMG000027_roi0_osteosarcoma_ok_iou0.83_conf0.959.jpg.png

  [02/20]  IMG000947_roi0_osteochondroma_ok_iou0.79_conf0.933.jpg  (GT: Benign)
   

In [29]:
import shutil

source_folder = "/kaggle/working/explainability_final"
zip_path = "/kaggle/working/explainability_final"

shutil.make_archive(
    base_name=zip_path.replace(".zip",""),
    format="zip",
    root_dir=source_folder
)

print("✅ Zipped successfully:", zip_path)


✅ Zipped successfully: /kaggle/working/explainability_final
